In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Construction Project Lifecycle Simulator v3
# MAGIC ### Suffolk Construction — Advanced Analytics & Data Science
# MAGIC
# MAGIC **Cell 1 of 5**: Config, imports, calibration table loading, indexed lookups.
# MAGIC
# MAGIC Reads from all profile/constraint tables built during synthetic data generation.
# MAGIC Uses the same hierarchical lookup patterns as your existing generators.

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1.1 Imports & Seeds

# COMMAND ----------

from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional
from datetime import date, timedelta
from collections import defaultdict
import numpy as np
import pandas as pd
import hashlib, json, calendar, time, math, random, requests
from scipy.stats import beta as beta_dist

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
STUDENT_T_DF = 4

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1.2 Table Paths

# COMMAND ----------

# ── Synthetic data (generated by your CTGAN/copula pipeline) ──
SYNTHETIC_PROJECT_META = "analysts.self_managed.synthetic_project_meta_data_profiled"
SYNTHETIC_SCHEDULE_BL  = "analysts.self_managed.synthetic_schedule_baseline_v1"
SYNTHETIC_TASK_SUMMARY = "analysts.self_managed.synthetic_task_summary_by_sow_v1"
SYNTHETIC_RFI          = "analysts.self_managed.synthetic_rfi_v1"
REBASELINE_METADATA    = "analysts.self_managed.rebaseline_event_metadata_v1"

# ── Profile & constraint tables (from your calibration work) ──
SCHEDULE_PROFILE     = "analysts.self_managed.schedule_baseline_profile"
SCHEDULE_CONSTRAINTS = "analysts.self_managed.schedule_baseline_constraints"
TASK_PROFILE         = "analysts.self_managed.task_summary_profile"
TASK_CONSTRAINTS     = "analysts.self_managed.task_summary_constraints"
SOW_FREQUENCY_TABLE  = "analysts.self_managed.task_summary_sow_frequency"
SOW_OVERLAP_TABLE    = "analysts.self_managed.task_summary_sow_overlaps"
COMPLETION_PROG_TABLE = "analysts.self_managed.task_summary_completion_progression"
DATE_SHIFT_TABLE     = "analysts.self_managed.task_summary_date_shift_profiles"
RFI_PROFILE          = "analysts.self_managed.rfi_profile"
RFI_CONSTRAINTS      = "analysts.self_managed.rfi_constraints"
BUDGET_PROFILE       = "analysts.self_managed.budget_profile_v1"
BUDGET_CONSTRAINTS   = "analysts.self_managed.budget_constraints_v1"

# ── Output tables (5 separate Delta tables) ──
OUTPUT_CATALOG = "analysts.self_managed"
OUT_SCHEDULE_SNAPSHOTS = f"{OUTPUT_CATALOG}.sim_schedule_snapshots"
OUT_BUDGET_SNAPSHOTS   = f"{OUTPUT_CATALOG}.sim_budget_snapshots"
OUT_RFI_EVENTS         = f"{OUTPUT_CATALOG}.sim_rfi_events"
OUT_CHANGE_ORDERS      = f"{OUTPUT_CATALOG}.sim_change_orders"
OUT_RUN_REGISTRY       = f"{OUTPUT_CATALOG}.sim_run_registry"

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1.3 Load All Calibration Tables

# COMMAND ----------

print("Loading calibration tables...")

def _parse_json(j):
    if pd.isna(j): return {}
    try: return json.loads(j)
    except: return {}

# Schedule baseline
sched_profile_df = spark.table(SCHEDULE_PROFILE).toPandas()
sched_constr_df  = spark.table(SCHEDULE_CONSTRAINTS).toPandas()
sched_constr_df['bounds'] = sched_constr_df['bounds_json'].apply(_parse_json)
print(f"  Schedule: {len(sched_profile_df)} profile, {len(sched_constr_df)} constraint rows")

# Task summary
task_profile_df  = spark.table(TASK_PROFILE).toPandas()
task_constr_df   = spark.table(TASK_CONSTRAINTS).toPandas()
task_constr_df['bounds'] = task_constr_df['bounds_json'].apply(_parse_json)
sow_freq_df      = spark.table(SOW_FREQUENCY_TABLE).toPandas()
sow_overlap_df   = spark.table(SOW_OVERLAP_TABLE).toPandas()
print(f"  Task: {len(task_profile_df)} profile, {len(task_constr_df)} constraint, "
      f"{len(sow_freq_df)} SOW freq, {len(sow_overlap_df)} overlap rows")

# Completion progression & date shifts (from your task summary generator)
try:
    completion_prog_df = spark.table(COMPLETION_PROG_TABLE).toPandas()
    date_shift_df      = spark.table(DATE_SHIFT_TABLE).toPandas()
    print(f"  Completion progression: {len(completion_prog_df)} rows, Date shifts: {len(date_shift_df)} rows")
except Exception as e:
    completion_prog_df = pd.DataFrame()
    date_shift_df = pd.DataFrame()
    print(f"  ⚠ Completion/date shift tables not found: {e}")

# RFI
rfi_profile_df = spark.table(RFI_PROFILE).toPandas()
rfi_constr_df  = spark.table(RFI_CONSTRAINTS).toPandas()
rfi_constr_df['bounds'] = rfi_constr_df['bounds_json'].apply(_parse_json)
print(f"  RFI: {len(rfi_profile_df)} profile, {len(rfi_constr_df)} constraint rows")

# Budget
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")
budget_profile_df = spark.table(BUDGET_PROFILE).toPandas()
budget_constr_df  = spark.table(BUDGET_CONSTRAINTS).toPandas()
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
print(f"  Budget: {len(budget_profile_df)} profile, {len(budget_constr_df)} constraint rows")

# Rebaseline metadata
rebaseline_meta_df = spark.table(REBASELINE_METADATA).toPandas()
rebaseline_meta_df['dt_rebaseline_occurred'] = pd.to_datetime(rebaseline_meta_df['dt_rebaseline_occurred'])
print(f"  Rebaseline events: {len(rebaseline_meta_df)}")

# SOW completion from synthetic (gap-fill)
sow_completion_df = spark.sql(f"""
    SELECT cat_sow, avg(n_tasks_total) AS avg_tasks, stddev(n_tasks_total) AS task_std,
           avg(pct_complete) AS avg_pct_complete
    FROM {SYNTHETIC_TASK_SUMMARY} WHERE is_active_record = true GROUP BY cat_sow
""").toPandas()
print(f"  SOW completion profiles: {len(sow_completion_df)} SOWs (synthetic gap-fill)")

print("✓ All calibration tables loaded")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1.4 Build Indexed Lookups

# COMMAND ----------

# ═══════════════════════════════════════════════════════════════
# HIERARCHICAL LOOKUP — same pattern as all your generators
# ═══════════════════════════════════════════════════════════════

def hier_lookup(df, field_name, sbu=None, size_cat=None, contract_bucket=None,
                asset_class=None, cat_sow=None, div=None, cat=None, min_n=3):
    """Hierarchical profile lookup with fallback chain."""
    lookups = [
        ('SBU_CONTRACT', {'cat_sbu': sbu, 'contract_bucket': contract_bucket}),
        ('SBU_SIZE',     {'cat_sbu': sbu, 'size_category': size_cat}),
        ('SBU_ASSET',    {'cat_sbu': sbu, 'cat_main_asset_class': asset_class}),
        ('SOW_SBU',      {'cat_sbu': sbu, 'cat_sow': cat_sow}),
        ('SOW_TYPE',     {'cat_sow': cat_sow}),
        ('SBU',          {'cat_sbu': sbu}),
        ('GLOBAL',       {}),
    ]
    base = df[df['field'] == field_name]
    if div is not None and 'cat_budget_division' in base.columns:
        base = base[base['cat_budget_division'] == div]
    if cat is not None and 'cat_budget_category' in base.columns:
        base = base[base['cat_budget_category'] == cat]
    for level, filters in lookups:
        mask = base['level'] == level
        for col, val in filters.items():
            if val is not None and col in base.columns:
                mask &= (base[col] == val)
        rows = base[mask]
        if len(rows) > 0:
            row = rows.iloc[0]
            n = row.get('n_projects', None)
            if pd.notna(n) and n >= min_n:
                return row
            elif level in ('SBU', 'GLOBAL'):
                return row
    return None

def hier_lookup_bounds(df, field_name, sbu=None, size_cat=None, contract_bucket=None, cat_sow=None):
    """Hierarchical constraint bounds lookup."""
    lookups = [
        ('SOW_SBU',      {'cat_sbu': sbu, 'cat_sow': cat_sow}),
        ('SOW_TYPE',     {'cat_sow': cat_sow}),
        ('SBU_CONTRACT', {'cat_sbu': sbu, 'contract_bucket': contract_bucket}),
        ('SBU_SIZE',     {'cat_sbu': sbu, 'size_category': size_cat}),
        ('SBU',          {'cat_sbu': sbu}),
        ('GLOBAL',       {}),
        ('TIMING',       {}),
    ]
    base = df[df['field'] == field_name]
    for level, filters in lookups:
        mask = base['level'] == level
        for col, val in filters.items():
            if val is not None and col in base.columns:
                mask &= (base[col] == val)
        rows = base[mask]
        if len(rows) > 0:
            return rows.iloc[0]['bounds']
    return {}

def sample_from_bounds(bounds, use_triangular=True):
    """Sample from profiled bounds with triangular distribution."""
    if not bounds: return None
    p25 = float(bounds.get('p25', bounds.get('median', bounds.get('p50', 0))))
    p50 = float(bounds.get('median', bounds.get('p50', 0)))
    p75 = float(bounds.get('p75', p50))
    if p25 == p50 == p75: return p50
    if p25 >= p50: p25 = max(0, p50 - 0.5)
    if p75 <= p50: p75 = p50 + 0.5
    val = np.random.triangular(p25, p50, p75) if use_triangular else np.random.uniform(p25, p75)
    val *= (1 + np.random.uniform(-0.05, 0.05))  # jitter
    p95 = bounds.get('p95')
    if p95 is not None and pd.notna(p95): val = min(val, float(p95))
    return val

def soft_clamp(val, p10, p90, p99):
    """Soft clamp from your budget generator."""
    if val > p99: return p99
    if val > p90:
        excess = (val - p90) / (p99 - p90 + 1e-9)
        return p90 + (p99 - p90) * (excess ** 0.5)
    return max(val, p10)

def safe_triangular(p25, median, p75):
    """np.random.triangular safe — handles zero-variance."""
    p25 = min(float(p25), float(median))
    p75 = max(float(p75), float(median))
    if p25 == median == p75: return float(median)
    return np.random.triangular(p25, median, p75)

# ═══════════════════════════════════════════════════════════════
# INDEX: SOW frequency by SBU
# ═══════════════════════════════════════════════════════════════

sow_freq_index = {}
for sbu_val, grp in sow_freq_df.groupby('cat_sbu'):
    grp = grp.copy()
    grp['prob'] = grp['pct_projects'] / grp['pct_projects'].sum()
    sow_freq_index[sbu_val] = {'sows': grp['cat_sow'].values, 'probs': grp['prob'].values}

# ═══════════════════════════════════════════════════════════════
# INDEX: SOW overlaps (from your task summary generator)
# ═══════════════════════════════════════════════════════════════

overlap_index = {}
for _, row in sow_overlap_df.sort_values('n_transitions', ascending=False).iterrows():
    key = (row['level'], row['sow_pair'])
    if key not in overlap_index:
        overlap_index[key] = {
            'p25': float(row['p25_pct']), 'median': float(row['median_pct']),
            'p75': float(row['p75_pct']),
        }

# ═══════════════════════════════════════════════════════════════
# INDEX: Completion progression (from your task summary generator)
# ═══════════════════════════════════════════════════════════════

completion_index = {}
if len(completion_prog_df) > 0:
    for _, row in completion_prog_df.iterrows():
        key = (row['level'], row.get('cat_sbu'),
               row['project_progress_bucket'], row['sow_position_bucket'])
        completion_index[key] = row

# ═══════════════════════════════════════════════════════════════
# INDEX: Date shift profiles (from your task summary generator)
# ═══════════════════════════════════════════════════════════════

date_shift_index = {}
if len(date_shift_df) > 0:
    for _, row in date_shift_df.iterrows():
        date_shift_index[(row['completion_bucket'], row['position_bucket'])] = row

# ═══════════════════════════════════════════════════════════════
# INDEX: Avg task duration by SOW (from task profiles)
# ═══════════════════════════════════════════════════════════════

avg_duration_index = {}
for _, row in task_profile_df[task_profile_df['field'] == 'avg_task_duration'].iterrows():
    if pd.notna(row.get('cat_sow')):
        avg_duration_index[row['cat_sow']] = {
            'p25': float(row['p25']) if pd.notna(row.get('p25')) else float(row['median']),
            'median': float(row['median']),
            'p75': float(row['p75']) if pd.notna(row.get('p75')) else float(row['median']),
        }

# ═══════════════════════════════════════════════════════════════
# INDEX: RFI profiles — trade weights, lifecycle timing, status, design issue
# ═══════════════════════════════════════════════════════════════

rfi_trade_weights = {}
rfi_lifecycle_timing = {}
rfi_status_dist = {}
rfi_design_issue_pct = {}
rfi_sow_timing_probs = {}

for _, row in rfi_profile_df.iterrows():
    f = row['field']
    if f == 'trade_weight':
        rfi_trade_weights.setdefault(row.get('cat_sbu') or 'GLOBAL', {})[row['cat_trade_l1']] = float(row.get('pct_of_sbu', 0))
    elif f == 'lifecycle_timing_pct':
        rfi_lifecycle_timing[row['cat_trade_l1']] = {
            'p10': float(row.get('p10', 20)), 'p25': float(row.get('p25', 35)),
            'median': float(row.get('median', 50)), 'p75': float(row.get('p75', 70)),
            'p90': float(row.get('p90', 85)),
        }
    elif f == 'status_distribution':
        rfi_status_dist.setdefault(row.get('cat_sbu') or 'GLOBAL', {})[row['status_simplified']] = float(row['pct'])
    elif f == 'design_issue_pct':
        rfi_design_issue_pct[row['cat_trade_l1']] = float(row.get('pct_design_issue', 35)) / 100.0
    elif f == 'sow_timing_zone_pct':
        rfi_sow_timing_probs.setdefault(row['cat_trade_l1'], {})[row['status_simplified']] = float(row['pct_of_trade'])
    elif f == 'l3_frequency':
        pass  # will use in full diversity engine later

# RFI constraints
_res = rfi_constr_df[(rfi_constr_df['field'] == 'resolution_days') & (rfi_constr_df['level'] == 'GLOBAL')]
rfi_resolution_bounds = _res.iloc[0]['bounds'] if len(_res) > 0 else {'p25': 4, 'median': 10, 'p75': 21, 'p90': 49}

_rb_spike = rfi_constr_df[rfi_constr_df['field'] == 'rebaseline_spike_ratio']
rb_spike_bounds = _rb_spike.iloc[0]['bounds'] if len(_rb_spike) > 0 else {'p25': 0.79, 'median': 1.08, 'p75': 1.58}

def get_rfi_count_bounds(sbu, size_cat):
    for level, filters in [
        ('SBU_SIZE', {'cat_sbu': sbu, 'size_category': size_cat}),
        ('SBU', {'cat_sbu': sbu}), ('GLOBAL', {}),
    ]:
        rows = rfi_constr_df[(rfi_constr_df['field'] == 'n_rfis_per_project') & (rfi_constr_df['level'] == level)]
        for col, val in filters.items():
            if val and col in rows.columns: rows = rows[rows[col] == val]
        if len(rows) > 0: return rows.iloc[0]['bounds']
    return {'p25': 50, 'median': 200, 'p75': 500, 'p90': 1000}

# ═══════════════════════════════════════════════════════════════
# INDEX: Budget profiles — S-curves, CO rates, division presence,
#        category weights, cost code frequencies, decomposition ratios
# ═══════════════════════════════════════════════════════════════

budget_scurves = {}
budget_co_rates = {}
budget_div_presence = {}
budget_cat_weights = {}
budget_code_lookup = {}
budget_code_counts = {}
budget_decomp_ratios = {}

for _, r in budget_constr_df.iterrows():
    f = r.get('field', '')
    jb = r.get('json_blob')
    if f == 'scurve_shape' and pd.notna(jb):
        budget_scurves[r['cat_budget_division']] = json.loads(jb)
    elif f == 'change_order_rate' and pd.notna(jb):
        key = (r['cat_budget_division'], r.get('cat_sbu') or 'GLOBAL')
        budget_co_rates[key] = json.loads(jb)
    elif f == 'cost_code_frequency_table' and pd.notna(jb):
        budget_code_lookup[r['cat_budget_division']] = json.loads(jb)
    elif f == 'n_cost_codes_per_division':
        key = (r['cat_budget_division'], r.get('cat_sbu') or 'GLOBAL')
        budget_code_counts[key] = r.to_dict()

for _, r in budget_profile_df.iterrows():
    f = r.get('field', '')
    if f == 'division_presence_prob' and pd.notna(r.get('str_value')):
        budget_div_presence[(r['cat_budget_division'], r.get('cat_sbu') or 'GLOBAL')] = float(r['str_value'])
    elif f == 'category_weight_vector' and pd.notna(r.get('json_blob')):
        budget_cat_weights[r['cat_budget_division']] = json.loads(r['json_blob'])
    elif f == 'cost_decomposition_ratios' and pd.notna(r.get('json_blob')):
        key = (r['cat_budget_division'], r.get('cat_budget_category'))
        budget_decomp_ratios[key] = json.loads(r['json_blob'])

# ═══════════════════════════════════════════════════════════════
# DOMAIN CONSTANTS (from your generators)
# ═══════════════════════════════════════════════════════════════

# Trade → SOW mapping (from your RFI generator)
TRADE_TO_SOW = {
    'A SUBSTRUCTURE': ['(13) Below Grade Structure', '(13) Foundations'],
    'B SHELL':        ['(14) Superstructure', '(17) Facade', '(18) Roofing'],
    'C INTERIORS':    ['(25) Interior Rough', '(26) Interior Finishes', '(27) Worklist'],
    'D SERVICES':     ['(29) Fire Life Safety', '(30) Systems Start-up', '(08) Permanent Power'],
    'E EQUIPMENT AND FURNISHINGS': ['(24) Remaining Elevator', '(21) Hoist'],
    'F SPECIAL CONSTRUCTION AND DEMOLITION': ['(12) Initial Site Work', '(12C) Demo & Abatement'],
    'G SITEWORK':     ['(28) Final Site Work', '(12) Initial Site Work'],
    'Z GENERAL':      ['(10) All Submittals', '(33) NO CODE', '(09) Buys'],
}

# SOW → Division mapping (from your budget generator)
SOW_TO_DIVISIONS = {
    "(13) Below Grade Structure + Foundations": ["31", "03"],
    "(14) Superstructure (Concrete Structure)": ["03", "05"],
    "(14A) Steel Erection": ["05"], "(14B) Deck & Detail Steel": ["05"],
    "(14C) Concrete on Metal Decks": ["03"],
    "(17) Facade (Weathertight)": ["07", "08"], "(17B) Facade - Complete": ["07", "08"],
    "(18) Roofing (Weathertight)": ["07"],
    "(25) Interior Rough": ["06", "09"], "(26) Interior Finishes": ["09", "10", "12"],
    "(12) Initial Site Work": ["02", "32"], "(12A) Enabling": ["02"],
    "(28) Final Site Work": ["32", "33"],
    "(30) Systems Start-up & Testing": ["21", "22", "23", "26"],
    "(08) Permanent Power Distributed": ["26"],
    "(10B) MEP Coordination": ["21", "22", "23", "26"],
    "(09) Buys": ["98", "99"], "(10) All Submittals": ["98"],
    "(10A) Precon- Design & Permit": ["98", "99"],
    "(27) Worklist & Punch list": ["09", "10"],
    "(29) Fire Life Safety": ["21"],
    "(31) Final Inspections & Close-out": ["98", "99", "01"],
    "(33) NO CODE (All remaining activities)": ["95"],
}

DIV_TO_SOWS = {}
for sow, divs in SOW_TO_DIVISIONS.items():
    for d in divs:
        DIV_TO_SOWS.setdefault(d, []).append(sow)

# Division lifecycle windows (from your budget generator)
DIV_LIFECYCLE_WINDOW = {
    "02":(0.0,0.3),"31":(0.0,0.3),"03":(0.0,0.5),"04":(0.0,0.5),"05":(0.1,0.5),
    "06":(0.3,0.8),"07":(0.2,0.6),"08":(0.2,0.6),"09":(0.4,1.0),"10":(0.5,1.0),
    "11":(0.3,0.8),"12":(0.5,1.0),"13":(0.0,0.4),"14":(0.3,0.8),
    "21":(0.2,0.8),"22":(0.2,0.8),"23":(0.2,0.8),"26":(0.2,0.8),
    "27":(0.3,0.9),"28":(0.3,0.9),"32":(0.0,0.3),"33":(0.0,0.4),
    "95":(0.0,1.0),"98":(0.0,1.0),"99":(0.0,1.0),"01":(0.0,1.0),
}

# Category lifecycle offsets (from your budget generator)
CATEGORY_LIFECYCLE_OFFSET = {
    "materials": -0.16, "labor": -0.12, "lit": -0.09, "glpd": -0.01,
    "subcontractor": 0.0, "other": 0.0, "ccip": +0.08, "subguard": +0.10,
    "contingency": +0.05, "fee": 0.0, "allowance": +0.41,
    "liberty builds": +0.16, "liberty equipment": +0.28,
}

MANDATORY_DIVISIONS = {"98", "99", "01"}

# Due date options (from your RFI generator, weighted toward 7-day Suffolk standard)
DUE_DATE_OPTIONS = [5, 7, 7, 7, 10, 14, 21]

# Delay category PMF (from your schedule generator)
DELAY_CATEGORY_PMF = {
    'minimal': 0.093, 'small': 0.313, 'moderate': 0.232,
    'major': 0.179, 'large': 0.081, 'severe': 0.102,
}

# CO probabilities (from your budget generator)
CO_PROB_REBASE_MONTH = 0.8227
CO_PROB_NORMAL_MONTH = 0.6261
DELAY_CO_MULTIPLIER = {
    'minimal': 1.0, 'small': 1.5, 'moderate': 2.5,
    'major': 3.5, 'large': 5.0, 'severe': 7.0,
}

# SOW overrun profiles (from your task summary generator)
SOW_OVERRUN_PROFILES = {
    '(31) Final Inspections & Close-out':      (0.5714, 62,  169, 443),
    '(33) NO CODE (All remaining activities)': (0.5619, 77,  184, 471),
    '(27) Worklist & Punch list':              (0.5083, 66,  168, 316),
    '(26) Interior Finishes':                  (0.3333, 90,  244, 567),
    '(30) Systems Start-up & Testing':         (0.3301, 79,  234, 462),
    '(28) Final Site Work':                    (0.2695, 118, 269, 751),
    '(29) Fire Life Safety':                   (0.2414, 151, 304, 795),
    '(25) Interior Rough':                     (0.1814, 174, 408, 831),
    '(17B) Facade - Complete':                 (0.1400, 62,  421, 819),
}

# SOW min durations (from your task summary generator)
SOW_MIN_DURATIONS = {
    '(13) Below Grade Structure + Foundations': 30,
    '(14) Superstructure (Concrete Structure)': 21,
    '(14A) Steel Erection': 21, '(14B) Deck & Detail Steel': 14,
    '(14C) Concrete on Metal Decks': 14,
    '(17) Facade (Weathertight)': 21, '(17B) Facade - Complete': 21,
    '(18) Roofing (Weathertight)': 14,
    '(25) Interior Rough': 30, '(26) Interior Finishes': 30,
    '(29) Fire Life Safety': 14, '(30) Systems Start-up & Testing': 14,
    '(08) Permanent Power Distributed': 14,
    '(12) Initial Site Work': 14, '(12A) Enabling': 14,
    '(28) Final Site Work': 14,
    '(31) Final Inspections & Close-out': 7, '(27) Worklist & Punch list': 7,
}
SOW_MIN_DURATION_DEFAULT = 7

# SOW critical path profiles (from your task summary generator)
SOW_CRITICAL_PATH_PROFILES = {
    '(26) Interior Finishes': (2.17, 0.8), '(25) Interior Rough': (2.32, 0.9),
    '(13) Below Grade Structure + Foundations': (4.81, 1.5),
    '(14) Superstructure (Concrete Structure)': (9.17, 2.0),
    '(12) Initial Site Work': (6.91, 1.8), '(14A) Steel Erection': (6.47, 1.8),
    '(27) Worklist & Punch list': (1.86, 0.7),
    '(31) Final Inspections & Close-out': (8.45, 2.0),
    '(29) Fire Life Safety': (5.00, 1.5), '(30) Systems Start-up & Testing': (2.59, 0.9),
    '(14B) Deck & Detail Steel': (5.52, 1.5), '(18) Roofing (Weathertight)': (2.53, 0.9),
    '(17) Facade (Weathertight)': (3.04, 1.0), '(14C) Concrete on Metal Decks': (4.02, 1.3),
    '(28) Final Site Work': (3.44, 1.1), '(17B) Facade - Complete': (1.13, 0.5),
}

MEDIAN_DELAY_DAYS = 41.0  # from your task summary generator

print(f"\n✓ Indexed: {len(sow_freq_index)} SBU SOW freq, {len(overlap_index)} overlaps, "
      f"{sum(len(v) for v in rfi_trade_weights.values())} trade weights, "
      f"{len(budget_scurves)} budget S-curves, {len(budget_co_rates)} CO rates, "
      f"{len(completion_index)} completion progression entries, {len(date_shift_index)} date shift entries")

✓ Config, State & Helpers loaded


In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC ## Cell 2: State Vector, Helpers, Initializer
# MAGIC Full SOW placement with overlap/overrun/scaling logic from your task summary generator.

# COMMAND ----------

# MAGIC %md
# MAGIC ### 2.1 State Vector

# COMMAND ----------

@dataclass
class SOWState:
    cat_sow: str
    sow_sequence_order: int
    is_active: bool = False
    pct_complete: float = 0.0
    n_tasks_total: int = 0
    n_tasks_completed: int = 0
    n_tasks_active: int = 0
    n_tasks_not_started: int = 0
    dt_sow_start: Optional[date] = None
    dt_sow_finish: Optional[date] = None
    planned_weight: float = 0.0
    sow_duration_days: int = 0
    # For drift damping across rebaselines (from your task summary generator)
    v0_sow_start: Optional[date] = None
    n_tasks_base: int = 0  # original task count for perturbation

@dataclass
class ShockEffect:
    shock_type: str
    months_remaining: int
    spi_penalty: float = 0.0
    bpi_penalty: float = 0.0
    rfi_multiplier: float = 1.0

@dataclass
class BudgetLineState:
    """Per-division budget tracking (from your budget generator)."""
    division: str
    cost_code: str
    full_cost_code: str
    category: str
    amt_original: float = 0.0
    cum_approved_in: float = 0.0
    cum_approved_out: float = 0.0
    jtd_max: float = 0.0  # monotonically non-decreasing

@dataclass
class MonthlyState:
    sim_month: int = 0
    dt_snapshot: Optional[date] = None
    # Schedule
    pct_complete: float = 0.0
    spi: float = 1.0
    days_behind: int = 0
    current_tco: Optional[date] = None
    original_tco: Optional[date] = None
    pct_float_consumed: float = 0.0
    # Rebaseline
    n_rebaselines: int = 0
    n_rebaselines_max: int = 0
    months_since_rebaseline: Optional[int] = None
    is_rebaseline_month: bool = False
    cat_schedule_code: str = ""
    rebaseline_months: List[int] = field(default_factory=list)
    rebaseline_delay_category: Optional[str] = None  # for CO modulation
    # SOW
    sow_states: List[SOWState] = field(default_factory=list)
    # Budget
    bpi: float = 1.0
    planned_value: float = 0.0
    earned_value: float = 0.0
    actual_cost: float = 0.0
    amt_budget_total: float = 0.0
    amt_contract: float = 0.0
    eac: float = 0.0
    cumulative_spend: float = 0.0
    monthly_burn: float = 0.0
    budget_lines: List[BudgetLineState] = field(default_factory=list)
    # Change orders
    n_approved_cos: int = 0
    amt_approved_co_value: float = 0.0
    n_pending_cos: int = 0
    amt_pending_co_value: float = 0.0
    # RFI
    open_rfis: int = 0
    total_rfis: int = 0
    rfi_rate_3m: float = 0.0
    rfi_history: List[int] = field(default_factory=list)
    # Risk
    composite_risk: float = 0.0
    active_shocks: List[ShockEffect] = field(default_factory=list)
    # Phi (internal, not exposed)
    phi: float = 0.5

    def compute_composite_risk(self):
        exp = max(1, self.rfi_rate_3m * 3)
        self.composite_risk = max(0.0, min(1.0,
            0.4 * max(0, 1.0 - self.spi) +
            0.4 * max(0, 1.0 - self.bpi) +
            0.2 * min(1.0, self.open_rfis / exp)
        ))
        return self.composite_risk

    def get_spi_state_index(self):
        if self.spi < 0.85: return 0
        elif self.spi < 0.92: return 1
        elif self.spi < 1.0: return 2
        elif self.spi < 1.08: return 3
        else: return 4

    def apply_shock_effects(self):
        still = []
        for s in self.active_shocks:
            self.spi = max(0.5, self.spi - s.spi_penalty)
            self.bpi = max(0.5, self.bpi - s.bpi_penalty)
            s.months_remaining -= 1
            if s.months_remaining > 0: still.append(s)
        self.active_shocks = still

    def get_rfi_shock_multiplier(self):
        return math.prod(s.rfi_multiplier for s in self.active_shocks) if self.active_shocks else 1.0

# COMMAND ----------

# MAGIC %md
# MAGIC ### 2.2 Helper Functions

# COMMAND ----------

def add_months(d, months):
    month = d.month - 1 + months; year = d.year + month // 12; month = month % 12 + 1
    day = min(d.day, calendar.monthrange(year, month)[1])
    return date(year, month, day)

def days_between(d1, d2): return (d2 - d1).days
def generate_schedule_code(pid, v): return f"{pid}-000.01-BL" if v == 0 else f"{pid}-{v:03d}.MO-RB"
def generate_rfi_id(pid, seq): return f"{pid}-RFI-{seq:04d}"
def generate_co_id(pid, seq): return f"{pid}-CO-{seq:04d}"
def generate_run_id(pid, idx, seed): return f"{pid}-RUN-{idx:02d}-S{seed}"
def hash_metadata(m): return hashlib.md5(json.dumps(m, sort_keys=True, default=str).encode()).hexdigest()[:12]

def sample_student_t(rng, mu, sigma, df=4):
    return mu + sigma * rng.standard_t(df)

def sample_truncated_student_t(rng, mu, sigma, lo, hi, df=4):
    for _ in range(100):
        v = sample_student_t(rng, mu, sigma, df)
        if lo <= v <= hi: return v
    return np.clip(mu, lo, hi)

def phi_modulate(base, phi, fav=0.8, unfav=1.2):
    return base * (unfav + phi * (fav - unfav))

def scurve_planned_pct(month, total, alpha=2.5, beta_param=2.5):
    t = min(1.0, max(0.0, month / max(1, total)))
    return float(beta_dist.cdf(t, alpha, beta_param))

def interpolate_budget_scurve(div, lp):
    """Cumulative spend fraction using your profiled S-curve shapes per division."""
    shape = budget_scurves.get(div)
    if not shape: return min(1.0, lp)
    deciles = sorted([float(k) for k in shape.keys()])
    p50s = [shape[str(round(d, 1))]["p50"] for d in deciles]
    total = sum(p50s)
    if total <= 0: return min(1.0, lp)
    cum = 0.0
    for i, d in enumerate(deciles):
        if d > lp:
            if i > 0:
                frac = (lp - deciles[i-1]) / (d - deciles[i-1] + 1e-9)
                cum += p50s[i] * frac
            break
        cum += p50s[i]
    return min(1.0, cum / total)

def get_size_category(area):
    if area < 50000: return "Small"
    elif area < 100000: return "Medium"
    elif area < 200000: return "Large"
    else: return "XLarge"

def get_contract_bucket(amt):
    if amt < 50e6: return "Medium"
    elif amt < 100e6: return "Large"
    elif amt < 200e6: return "XLarge"
    else: return "XXLarge"

# Bucketing helpers (from your task summary generator)
def get_progress_bucket(timing):
    if timing < 0.2: return '0-20'
    elif timing < 0.4: return '20-40'
    elif timing < 0.6: return '40-60'
    elif timing < 0.8: return '60-80'
    else: return '80-100'

def get_position_bucket(normalized_pos):
    if normalized_pos < 0.2: return 'early'
    elif normalized_pos < 0.4: return 'early_mid'
    elif normalized_pos < 0.6: return 'mid'
    elif normalized_pos < 0.8: return 'mid_late'
    else: return 'late'

def get_completion_bucket(pct):
    if pct >= 95: return 'completed'
    elif pct >= 50: return 'mostly_done'
    elif pct >= 10: return 'in_progress'
    else: return 'not_started'

def get_date_shift_position_bucket(normalized_pos):
    if normalized_pos < 0.25: return 'early'
    elif normalized_pos < 0.50: return 'early_mid'
    elif normalized_pos < 0.75: return 'mid_late'
    else: return 'late'

def lookup_completion(sbu, progress_bucket, pos_bucket):
    """From your task summary generator — profiled completion % by progress × position."""
    for level_val, sbu_val in [('SBU', sbu), ('GLOBAL', None)]:
        row = completion_index.get((level_val, sbu_val, progress_bucket, pos_bucket))
        if row is not None:
            return safe_triangular(
                max(0.0, min(float(row['p25_complete']), float(row['median_complete']))),
                float(row['median_complete']),
                min(100.0, max(float(row['p75_complete']), float(row['median_complete'])))
            )
    return 0.0

def lookup_date_shift(completion_bucket, position_bucket):
    """From your task summary generator — date shifts by completion × position."""
    row = date_shift_index.get((completion_bucket, position_bucket))
    if row is not None:
        return {
            'finish_shift': int(safe_triangular(row['p25_finish_shift'], row['median_finish_shift'], row['p75_finish_shift'])),
            'start_shift': int(row['median_start_shift']),
            'duration_change': int(row['median_duration_change']),
        }
    return {'finish_shift': 0, 'start_shift': 0, 'duration_change': 0}

def estimate_dependency_metrics(seq_pos, n_total, cat_sow=None):
    """From your task summary generator — SOW-specific critical path + dependency metrics."""
    normalized = seq_pos / max(n_total, 1)
    # Critical path — SOW-specific first
    if cat_sow and cat_sow in SOW_CRITICAL_PATH_PROFILES:
        med, var = SOW_CRITICAL_PATH_PROFILES[cat_sow]
        pct_crit = np.clip(np.random.normal(med, var), max(0.5, med - 2*var), min(15.0, med + 2*var))
    else:
        if normalized < 0.25: pct_crit = np.random.uniform(3.0, 6.5)
        elif normalized < 0.50: pct_crit = np.random.uniform(4.0, 8.0)
        elif normalized < 0.75: pct_crit = np.random.uniform(2.0, 5.0)
        else: pct_crit = np.random.uniform(1.5, 4.0)
    # Predecessors/successors
    if normalized < 0.25: avg_pred, avg_succ = np.random.uniform(0.3,1.5), np.random.uniform(2.5,5.0)
    elif normalized < 0.50: avg_pred, avg_succ = np.random.uniform(1.0,2.5), np.random.uniform(1.5,3.5)
    elif normalized < 0.75: avg_pred, avg_succ = np.random.uniform(2.0,3.5), np.random.uniform(0.5,2.0)
    else: avg_pred, avg_succ = np.random.uniform(2.5,4.5), np.random.uniform(0.2,1.0)
    return round(avg_pred, 2), round(avg_succ, 2), round(pct_crit, 2)

def calculate_avg_task_duration(sow):
    """From your task summary generator — profiled avg task duration."""
    row = avg_duration_index.get(sow)
    if row: return round(max(1, safe_triangular(row['p25'], row['median'], row['p75'])), 1)
    return 30.0

# RFI sampling helpers (using your profiled distributions)
def sample_trade(sbu, rng):
    w = rfi_trade_weights.get(sbu, rfi_trade_weights.get('GLOBAL', {}))
    if not w: return 'Z GENERAL'
    trades, probs = list(w.keys()), np.array(list(w.values()))
    probs = probs / probs.sum()
    return trades[rng.choice(len(trades), p=probs)]

def sample_rfi_lifecycle_pct(trade, rng):
    """From your RFI generator — beta-shaped lifecycle timing per trade."""
    t = rfi_lifecycle_timing.get(trade)
    if not t: return float(rng.beta(2, 2) * 100)
    p10, med, p90 = float(t['p10']), float(t['median']), float(t['p90'])
    span = max(p90 - p10, 1)
    med_n = np.clip((med - p10) / span, 0.01, 0.99)
    iqr_n = max(0.05, (float(t['p75']) - float(t['p25'])) / span)
    conc = min(50, 0.5 / (iqr_n ** 2))
    return float(np.clip(p10 + rng.beta(max(0.5, med_n*conc), max(0.5, (1-med_n)*conc)) * span, p10, p90*1.1))

def sample_rfi_status(sbu, rng):
    dist = rfi_status_dist.get(sbu, rfi_status_dist.get('GLOBAL', {'closed': 0.92, 'open': 0.06, 'draft': 0.02}))
    statuses, probs = list(dist.keys()), np.array(list(dist.values()))
    probs = probs / probs.sum()
    return statuses[rng.choice(len(statuses), p=probs)]

def sample_resolution_days(rng):
    """From your RFI generator — lognormal resolution days."""
    b = rfi_resolution_bounds
    p25, med = float(b.get('p25', 4)), float(b.get('median', 10))
    p75, p90 = float(b.get('p75', 21)), float(b.get('p90', 49))
    sigma = (math.log(p75 + 1) - math.log(p25 + 1)) / (2 * 0.6745)
    mu = math.log(med + 1)
    return max(1, min(int(rng.lognormal(mu, sigma)), int(p90 * 1.15)))

def sample_n_rfis_for_project(sbu, size_cat, rng):
    """From your RFI generator — profiled project-level RFI count."""
    b = get_rfi_count_bounds(sbu, size_cat)
    p25, med, p75 = float(b.get('p25', 50)), float(b.get('median', 200)), float(b.get('p75', 500))
    sigma = (math.log(p75 + 1) - math.log(p25 + 1)) / (2 * 0.6745)
    mu = math.log(med + 1)
    return max(1, min(int(rng.lognormal(mu, sigma)), int(p75 * 1.15)))

# Budget helpers (from your budget generator)
def get_presence_prob(div, sbu):
    return budget_div_presence.get((div, sbu), budget_div_presence.get((div, 'GLOBAL'), 0.5))

def get_co_rate(div, sbu):
    return budget_co_rates.get((div, sbu), budget_co_rates.get((div, 'GLOBAL'),
           {'prob_co_in': 0.05, 'prob_co_out': 0.02, 'p50_co_in_ratio': 0.001,
            'p90_co_in_ratio': 0.005, 'p50_co_out_ratio': 0.001, 'p90_co_out_ratio': 0.003}))

def sample_n_codes(div, sbu):
    """From your budget generator."""
    key = (div, sbu)
    if key not in budget_code_counts: key = (div, 'GLOBAL')
    if key not in budget_code_counts: return 3
    r = budget_code_counts[key]
    p25, p75, p99 = r.get('p25') or 1, r.get('p75') or 5, r.get('p99') or 10
    return max(1, min(int(p99), int(np.random.normal((p25+p75)/2, (p75-p25)/2))))

def sample_cost_codes(div, n):
    codes = budget_code_lookup.get(div, [])
    if not codes: return [{"cat_cost_code": div, "cat_full_cost_code": f"{div} - unknown"}]
    w = np.array([c["n_projects"] for c in codes], dtype=float); w /= w.sum()
    chosen = np.random.choice(len(codes), size=min(n, len(codes)), replace=False, p=w)
    return [codes[i] for i in chosen]

def sample_budget_amount(div, cat, sbu, contract_bucket, asset_class, amt_contract):
    """From your budget generator — profiled amt_budget_original_ratio."""
    row = hier_lookup(budget_profile_df, 'amt_budget_original_ratio',
                      sbu=sbu, contract_bucket=contract_bucket, asset_class=asset_class,
                      div=div, cat=cat)
    if row is None: return amt_contract * 0.01
    p10 = row.get('p10') or 0.0; p50 = row.get('p50') or 0.01
    p90 = row.get('p90') or 0.05; p99 = row.get('p99') or 0.10
    val = np.random.lognormal(math.log(max(p50, 1e-9)),
                              (math.log(max(p90, 1e-9)) - math.log(max(p10, 1e-9))) / 4)
    return amt_contract * soft_clamp(val, p10, p90, p99)

# COMMAND ----------

# MAGIC %md
# MAGIC ### 2.3 Initializer (full SOW placement + budget setup)

# COMMAND ----------

class ProjectInitializer:
    """Initialize V0 state with full SOW placement logic from your task summary generator
    and division-level budget setup from your budget generator."""

    def __init__(self, project_id, cat_sbu, cat_region, cat_main_asset_class,
                 amt_contract, gross_area, dt_construction_start, planned_duration_months, rng):
        self.project_id = project_id; self.sbu = cat_sbu; self.region = cat_region
        self.asset_class = cat_main_asset_class; self.amt_contract = amt_contract
        self.gross_area = gross_area; self.dt_start = dt_construction_start
        self.planned_months = planned_duration_months; self.rng = rng
        self.size_cat = get_size_category(gross_area)
        self.contract_bucket = get_contract_bucket(amt_contract)

    def initialize(self):
        rng = self.rng
        phi = rng.beta(2.0, 2.0)

        # ── Rebaseline count (from your schedule profile) ──
        row = hier_lookup(sched_profile_df, 'n_rebaselines', sbu=self.sbu,
                          size_cat=self.size_cat, contract_bucket=self.contract_bucket)
        if row is not None and pd.notna(row.get('mean')):
            n_rb_max = min(int(rng.poisson(float(row['mean']))), int(row.get('p99', 6)))
        else:
            n_rb_max = int(rng.poisson(1.0))

        dt_tco = add_months(self.dt_start, self.planned_months)

        # ── SOW schedule (full placement from your task summary generator) ──
        sow_states = self._build_sow_schedule(dt_tco)

        # ── Division-level budget (from your budget generator) ──
        budget_lines, budget_total = self._build_budget()

        return MonthlyState(
            sim_month=0, dt_snapshot=self.dt_start, spi=1.0, bpi=1.0,
            current_tco=dt_tco, original_tco=dt_tco,
            n_rebaselines_max=n_rb_max,
            cat_schedule_code=generate_schedule_code(self.project_id, 0),
            sow_states=sow_states,
            amt_budget_total=budget_total, amt_contract=self.amt_contract,
            eac=budget_total, budget_lines=budget_lines, phi=phi,
        )

    def _build_sow_schedule(self, dt_tco):
        """Full SOW placement: frequency sampling → sequence → overlap → scaling → overruns."""
        rng = self.rng
        dt_start = self.dt_start
        total_days = max(1, (dt_tco - dt_start).days)

        # Sample SOW types from frequency table
        entry = sow_freq_index.get(self.sbu)
        if entry is None:
            entry = next(iter(sow_freq_index.values())) if sow_freq_index else None
        if entry is None: return []

        # Sample how many SOWs
        row = hier_lookup(task_profile_df, 'n_sows_per_project', sbu=self.sbu,
                          size_cat=self.size_cat, contract_bucket=self.contract_bucket)
        n_sows = int(np.clip(np.round(sample_from_bounds(
            {'p25': row.get('p25'), 'median': row.get('median'), 'p75': row.get('p75')}
        ) if row is not None else 20), 5, 40))

        n = min(n_sows, len(entry['sows']))
        chosen = list(rng.choice(entry['sows'], size=n, replace=False, p=entry['probs']))

        # Sample sequence positions (from your task summary constraints)
        sow_data = []
        for sow in chosen:
            # Anchor gravity for milestone SOWs
            if 'Notice to Proceed' in sow: pos = -100
            elif 'Substantial Completion' in sow: pos = 999
            elif 'GMP Execution' in sow: pos = -50
            else:
                bounds = hier_lookup_bounds(task_constr_df, 'sequence_position', sbu=self.sbu, cat_sow=sow)
                if bounds:
                    pos = int(np.clip(np.round(rng.normal(
                        float(bounds.get('mean', 10)), float(bounds.get('stddev', 3))
                    )), bounds.get('min', 1), bounds.get('max', 30)))
                else:
                    pos = int(rng.integers(1, 25))
            sow_data.append({'cat_sow': sow, 'pos': pos})
        sow_data.sort(key=lambda x: x['pos'])
        for i, sd in enumerate(sow_data, 1):
            sd['sequence_order'] = i

        # ── Place SOWs using overlap patterns (from your generator) ──
        dur_bounds = hier_lookup_bounds(task_constr_df, 'sow_duration_days',
                                         sbu=self.sbu, cat_sow=sow_data[0]['cat_sow'])
        first_dur = int(np.clip(np.round(sample_from_bounds(dur_bounds) if dur_bounds else 180), 30, 1000))
        sow_data[0]['dt_sow_start'] = dt_start
        sow_data[0]['sow_duration_days'] = first_dur
        sow_data[0]['dt_sow_finish'] = dt_start + timedelta(days=first_dur)

        for i in range(1, len(sow_data)):
            prev, curr = sow_data[i-1], sow_data[i]
            pair_key = ('PAIR', f"{prev['cat_sow']} → {curr['cat_sow']}")
            general_key = ('GENERAL', 'all_pairs')
            ov = overlap_index.get(pair_key, overlap_index.get(general_key, {'p25': 70, 'median': 84, 'p75': 95}))
            overlap_pct = np.clip(safe_triangular(float(ov['p25']), float(ov['median']), float(ov['p75'])), -50, 100) / 100.0
            offset = int(prev['sow_duration_days'] * (1 - overlap_pct))
            curr_start = prev['dt_sow_start'] + timedelta(days=max(0, offset))
            if curr_start > dt_tco + timedelta(days=60):
                curr_start = dt_tco - timedelta(days=60)
            dur_bounds = hier_lookup_bounds(task_constr_df, 'sow_duration_days',
                                             sbu=self.sbu, cat_sow=curr['cat_sow'])
            dur = int(np.clip(np.round(sample_from_bounds(dur_bounds) if dur_bounds else 180), 30, 1000))
            curr['dt_sow_start'] = curr_start
            curr['sow_duration_days'] = dur
            curr['dt_sow_finish'] = curr_start + timedelta(days=dur)

        # ── Proportional duration scaling (from your generator) ──
        latest_finish = max(sd['dt_sow_finish'] for sd in sow_data)
        if latest_finish > dt_tco:
            project_span = (dt_tco - dt_start).days
            actual_span = (latest_finish - dt_start).days
            if actual_span > 0 and project_span > 0:
                scale = max(project_span / actual_span, 0.50)
                for sd in sow_data:
                    offset_d = max(0, int((sd['dt_sow_start'] - dt_start).days * scale))
                    scaled_dur = max(SOW_MIN_DURATIONS.get(sd['cat_sow'], SOW_MIN_DURATION_DEFAULT),
                                     int(sd['sow_duration_days'] * scale))
                    sd['dt_sow_start'] = dt_start + timedelta(days=offset_d)
                    sd['sow_duration_days'] = scaled_dur
                    sd['dt_sow_finish'] = sd['dt_sow_start'] + timedelta(days=scaled_dur)

        # ── Stretch if SOWs finish too early (from your generator) ──
        latest = max(sd['dt_sow_finish'] for sd in sow_data)
        gap = (dt_tco - latest).days
        if gap > 60:
            current_span = (latest - dt_start).days
            target_span = (dt_tco - dt_start).days
            if current_span > 0:
                stretch = min(target_span / current_span, 2.5)
                for sd in sow_data:
                    off = int((sd['dt_sow_start'] - dt_start).days * stretch)
                    dur = max(SOW_MIN_DURATIONS.get(sd['cat_sow'], SOW_MIN_DURATION_DEFAULT),
                              int(sd['sow_duration_days'] * stretch))
                    sd['dt_sow_start'] = dt_start + timedelta(days=off)
                    sd['sow_duration_days'] = dur
                    sd['dt_sow_finish'] = sd['dt_sow_start'] + timedelta(days=dur)

        # ── Apply SOW overruns (from your generator) ──
        n_sows = len(sow_data)
        late_threshold = int(n_sows * 0.6)
        for i, sd in enumerate(sow_data):
            sow_name = sd['cat_sow']
            if sow_name in SOW_OVERRUN_PROFILES:
                prob, med_d, p75_d, p90_d = SOW_OVERRUN_PROFILES[sow_name]
                days_from_tco = (sd['dt_sow_finish'] - dt_tco).days
                if (days_from_tco > -30 or i >= late_threshold) and np.random.rand() < prob:
                    overrun = max(7, int(np.random.triangular(7, med_d, p75_d)))
                    sd['dt_sow_finish'] = dt_tco + timedelta(days=overrun)
                    sd['sow_duration_days'] = (sd['dt_sow_finish'] - sd['dt_sow_start']).days
                elif sd['dt_sow_finish'] > dt_tco:
                    if sd['dt_sow_start'] >= dt_tco:
                        sd['dt_sow_start'] = dt_tco - timedelta(days=30)
                    sd['dt_sow_finish'] = dt_tco
                    sd['sow_duration_days'] = (dt_tco - sd['dt_sow_start']).days
            elif sd['dt_sow_finish'] > dt_tco:
                if sd['dt_sow_start'] >= dt_tco:
                    sd['dt_sow_start'] = dt_tco - timedelta(days=30)
                sd['dt_sow_finish'] = dt_tco
                sd['sow_duration_days'] = (dt_tco - sd['dt_sow_start']).days

        # ── Build SOWState objects with task counts from constraints ──
        states = []
        for sd in sow_data:
            sow = sd['cat_sow']
            # Enforce min duration
            min_dur = SOW_MIN_DURATIONS.get(sow, SOW_MIN_DURATION_DEFAULT)
            if sd['sow_duration_days'] < min_dur:
                sd['sow_duration_days'] = min_dur
                sd['dt_sow_finish'] = sd['dt_sow_start'] + timedelta(days=min_dur)
            if sd['dt_sow_finish'] < sd['dt_sow_start']:
                sd['dt_sow_finish'] = sd['dt_sow_start'] + timedelta(days=min_dur)
                sd['sow_duration_days'] = min_dur

            # Milestone SOWs get small task counts
            if 'Notice to Proceed' in sow or 'Substantial Completion' in sow or 'GMP Execution' in sow:
                n_tasks = int(rng.integers(1, 10))
            else:
                task_bounds = hier_lookup_bounds(task_constr_df, 'n_tasks', sbu=self.sbu, cat_sow=sow)
                n_tasks = int(np.clip(np.round(sample_from_bounds(task_bounds) if task_bounds else 100), 5, 5000))

            dur_frac = sd['sow_duration_days'] / max(1, total_days)
            states.append(SOWState(
                cat_sow=sow, sow_sequence_order=sd['sequence_order'],
                n_tasks_total=n_tasks, n_tasks_base=n_tasks, n_tasks_not_started=n_tasks,
                dt_sow_start=sd['dt_sow_start'], dt_sow_finish=sd['dt_sow_finish'],
                sow_duration_days=sd['sow_duration_days'],
                planned_weight=dur_frac, v0_sow_start=sd['dt_sow_start'],
            ))

        # Normalize weights
        tw = sum(s.planned_weight for s in states)
        if tw > 0:
            for s in states: s.planned_weight /= tw
        return states

    def _build_budget(self):
        """Division-level budget setup from your budget generator."""
        rng = self.rng
        sbu, cb, asset = self.sbu, self.contract_bucket, self.asset_class
        amt = self.amt_contract

        # Determine which divisions are present
        present_divs = set(MANDATORY_DIVISIONS)
        for div in budget_scurves.keys():
            if div not in MANDATORY_DIVISIONS:
                if rng.random() < get_presence_prob(div, sbu):
                    present_divs.add(div)

        # Envelope cap from profile
        env_row = hier_lookup(budget_profile_df, 'total_budget_to_contract_ratio', sbu=sbu)
        env_cap = float(env_row['p90']) * amt if env_row is not None and pd.notna(env_row.get('p90')) else amt * 1.05

        # Build budget lines per division
        lines = []
        for div in present_divs:
            wd = budget_cat_weights.get(div, {"subcontractor": 1.0})
            cats = list(wd.keys()); wts = np.array(list(wd.values()), dtype=float); wts /= wts.sum()
            codes = sample_cost_codes(div, sample_n_codes(div, sbu))
            n_codes = len(codes)
            for code_row in codes:
                cat = np.random.choice(cats, p=wts)
                amt_orig = sample_budget_amount(div, cat, sbu, cb, asset, amt) / n_codes
                lines.append(BudgetLineState(
                    division=div, cost_code=code_row['cat_cost_code'],
                    full_cost_code=code_row['cat_full_cost_code'],
                    category=cat, amt_original=amt_orig,
                ))

        # Scale to envelope cap
        total_orig = sum(l.amt_original for l in lines)
        if total_orig > env_cap > 0:
            scale = env_cap / total_orig
            for l in lines: l.amt_original *= scale
            total_orig = env_cap

        return lines, total_orig

    def generate_shock_calendar(self, duration_months):
        """Pre-draw shock events with Student-t magnitudes."""
        cal = []; rng = self.rng
        for m in range(duration_months):
            cm = (self.dt_start.month + m - 1) % 12 + 1
            # Weather
            season_mult = {1:1.5,2:1.5,3:1.2,4:0.8,5:0.6,6:0.5,7:0.5,8:0.5,9:0.6,10:0.8,11:1.2,12:1.5}
            if rng.random() < 0.025 * season_mult.get(cm, 1.0):
                mag = max(0.03, min(0.10, sample_student_t(rng, 0.065, 0.02, STUDENT_T_DF)))
                cal.append({'month':m,'type':'weather','duration':int(rng.integers(1,4)),
                            'spi_penalty':mag,'bpi_penalty':0.0,'rfi_multiplier':1.0})
            # Design error
            de_prob = 0.015 * (2.0 if m/max(1,duration_months) < 0.30 else 1.0)
            if rng.random() < de_prob:
                mag = max(0.05, min(0.15, sample_student_t(rng, 0.10, 0.03, STUDENT_T_DF)))
                cal.append({'month':m,'type':'design_error','duration':1,
                            'spi_penalty':mag,'bpi_penalty':0.0,'rfi_multiplier':2.0})
            # Labor
            if rng.random() < 0.005:
                cal.append({'month':m,'type':'labor','duration':int(rng.integers(1,3)),
                            'spi_penalty':max(0.05, min(0.12, sample_student_t(rng, 0.085, 0.025, STUDENT_T_DF))),
                            'bpi_penalty':max(0.03, min(0.08, sample_student_t(rng, 0.055, 0.02, STUDENT_T_DF))),
                            'rfi_multiplier':1.0})
        return cal

# COMMAND ----------

print("✓ Cell 2 loaded: State, Helpers, Initializer")

✓ Engine loaded (Initializer + 7 PSUBs + ProjectSimulator)


In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC ## Cell 3: PSUBs 1-3 (Schedule, Shocks, Rebaseline)
# MAGIC Uses your completion progression tables, date shift profiles, and profiled delay distributions.

# COMMAND ----------

# MAGIC %md
# MAGIC ### PSUB 1 — Schedule & SOW Progress
# MAGIC Monthly progress from S-curve with SPI/RFI feedback dampening.
# MAGIC SOW-level updates use your completion progression table (progress_bucket × position_bucket).

# COMMAND ----------

def psub1_schedule(state, total_months, project_duration_days, rng):
    """PSUB 1: Schedule progress with feedback loops.
    
    Uses your profiled completion progression tables for SOW-level updates
    (the same lookup_completion function from your task summary generator).
    """
    # ── Expected monthly progress from S-curve ──
    pp_now = scurve_planned_pct(state.sim_month + 1, total_months)
    pp_prev = scurve_planned_pct(state.sim_month, total_months)
    mu = max(0.005, pp_now - pp_prev)
    sigma = mu * 0.35

    # ── Dampening factor D(t) — closed feedback loop ──
    # SPI component: poor SPI → slower progress
    spi_dampening = min(1.0, max(0.3, state.spi ** 0.5))

    # RFI backlog component: excess open RFIs → slower productivity
    exp_rfis = max(1.0, state.rfi_rate_3m * 3)
    theta_rfi = exp_rfis * 0.8
    rfi_excess = max(0, state.open_rfis - theta_rfi) / exp_rfis
    # Phi modulates absorption — good projects handle RFI backlog better
    gamma_h = phi_modulate(0.3, state.phi, 0.15, 0.45)
    rfi_dampening = max(0.4, 1.0 - gamma_h * rfi_excess)

    d_t = spi_dampening * rfi_dampening

    # ── Sample monthly progress (Student-t, fat-tailed) ──
    delta_pct = max(0.0, sample_student_t(rng, mu * d_t, sigma, STUDENT_T_DF))
    state.pct_complete = min(1.0, state.pct_complete + delta_pct)

    # ── Update SPI (with TPM autocorrelation from plan eq.6) ──
    planned_pct = scurve_planned_pct(state.sim_month + 1, total_months)
    state.planned_value = planned_pct * state.amt_budget_total
    if planned_pct > 0:
        raw_spi = min(2.0, state.pct_complete / planned_pct)
    else:
        raw_spi = 1.0

    # Blend with empirical TPM: SPI_t = 0.7*SPI_sampled + 0.3*E[SPI|prev_state]
    # TPM calibrated from real Platinum monthly snapshots (plan Section 6.1 eq.6)
    # Default mean-reverting TPM if not calibrated from data
    DEFAULT_TPM = np.array([
        [0.50, 0.25, 0.15, 0.07, 0.03],  # <0.85: tends to persist
        [0.20, 0.35, 0.25, 0.15, 0.05],  # 0.85-0.92
        [0.05, 0.15, 0.40, 0.25, 0.15],  # 0.92-1.0
        [0.03, 0.07, 0.20, 0.40, 0.30],  # 1.0-1.08
        [0.02, 0.05, 0.13, 0.30, 0.50],  # >1.08: tends to persist
    ])
    state_midpoints = [0.80, 0.885, 0.96, 1.04, 1.12]
    spi_state_idx = state.get_spi_state_index()
    tpm_row = DEFAULT_TPM[spi_state_idx]
    expected_spi = sum(p * m for p, m in zip(tpm_row, state_midpoints))
    blended_spi = 0.7 * raw_spi + 0.3 * expected_spi

    state.spi = max(0.50, min(1.50, blended_spi))

    # ── Update days behind ──
    if state.spi < 1.0:
        planned_days = state.sim_month * 30
        actual_days = planned_days / max(0.5, state.spi)
        state.days_behind = max(0, int(actual_days - planned_days))
    else:
        state.days_behind = max(0, state.days_behind - int(5 * (state.spi - 1.0) * 30))

    # ── Update float consumed ──
    if state.original_tco and state.dt_snapshot:
        slack = max(1, days_between(state.dt_snapshot, state.original_tco))
        state.pct_float_consumed = min(1.0, state.days_behind / slack)

    # ── SOW-level progress distribution ──
    # Uses your profiled completion progression if available,
    # falls back to proportional distribution
    dt_now = state.dt_snapshot
    if dt_now:
        n_sows = len(state.sow_states)
        # Determine project progress bucket (from your task summary generator)
        project_timing = state.sim_month / max(1, total_months)
        progress_bucket = get_progress_bucket(project_timing)

        for sow in state.sow_states:
            # Is this SOW active?
            sow.is_active = (sow.dt_sow_start and sow.dt_sow_finish and
                             sow.dt_sow_start <= dt_now <= sow.dt_sow_finish)

        active_sows = [s for s in state.sow_states if s.is_active and s.pct_complete < 1.0]

        if active_sows:
            # Try profiled completion first
            if completion_index:
                for sow in active_sows:
                    normalized_pos = sow.sow_sequence_order / max(n_sows, 1)
                    pos_bucket = get_position_bucket(normalized_pos)
                    profiled_pct = lookup_completion(state.__dict__.get('_sbu', 'commercial'),
                                                     progress_bucket, pos_bucket)
                    if profiled_pct > 0:
                        # Blend: 60% profiled progression + 40% S-curve-driven delta
                        target_pct = profiled_pct / 100.0
                        profiled_delta = max(0, target_pct - sow.pct_complete)
                        scurve_delta = delta_pct * (sow.planned_weight / max(0.01, sum(
                            s.planned_weight for s in active_sows
                        ))) * len(active_sows) * rng.uniform(0.85, 1.15)
                        sow_delta = 0.6 * profiled_delta + 0.4 * scurve_delta
                    else:
                        # Fall back to proportional
                        wts = [s.n_tasks_total * max(0.01, 1-s.pct_complete) * s.planned_weight for s in active_sows]
                        tw = sum(wts)
                        w = sow.n_tasks_total * max(0.01, 1-sow.pct_complete) * sow.planned_weight
                        sow_delta = delta_pct * (w/tw if tw > 0 else 0) * len(active_sows) * rng.uniform(0.85, 1.15)

                    sow.pct_complete = min(1.0, sow.pct_complete + max(0, sow_delta))
                    # Update task counts
                    sow.n_tasks_completed = int(sow.pct_complete * sow.n_tasks_total)
                    sow.n_tasks_not_started = max(0, sow.n_tasks_total - sow.n_tasks_completed - int(sow.n_tasks_total * 0.05))
                    sow.n_tasks_active = sow.n_tasks_total - sow.n_tasks_completed - sow.n_tasks_not_started
            else:
                # No completion progression table — pure proportional
                wts = [s.n_tasks_total * max(0.01, 1-s.pct_complete) * s.planned_weight for s in active_sows]
                tw = sum(wts)
                if tw > 0:
                    for sow, w in zip(active_sows, wts):
                        sd = delta_pct * (w/tw) * len(active_sows) * rng.uniform(0.85, 1.15)
                        sow.pct_complete = min(1.0, sow.pct_complete + sd)
                        sow.n_tasks_completed = int(sow.pct_complete * sow.n_tasks_total)
                        sow.n_tasks_not_started = max(0, sow.n_tasks_total - sow.n_tasks_completed - int(sow.n_tasks_total * 0.05))
                        sow.n_tasks_active = sow.n_tasks_total - sow.n_tasks_completed - sow.n_tasks_not_started

    return state

# COMMAND ----------

# MAGIC %md
# MAGIC ### PSUB 2 — Discrete Event Shocks

# COMMAND ----------

def psub2_shocks(state, shock_cal, rng):
    """PSUB 2: Fire pre-drawn shock events, apply continuing effects."""
    state.apply_shock_effects()
    for sh in [s for s in shock_cal if s['month'] == state.sim_month]:
        state.active_shocks.append(ShockEffect(
            sh['type'], sh['duration'], sh['spi_penalty'], sh['bpi_penalty'], sh['rfi_multiplier']
        ))
        state.spi = max(0.50, state.spi - sh['spi_penalty'])
        if sh['bpi_penalty'] > 0:
            state.bpi = max(0.50, state.bpi - sh['bpi_penalty'])
    return state

# COMMAND ----------

# MAGIC %md
# MAGIC ### PSUB 3 — Rebaseline Decision
# MAGIC SPI + BPI driven. Uses your profiled delay distributions per severity category
# MAGIC with slack-conditional capping (from your schedule generator).
# MAGIC SOW date shifts use your profiled date_shift_profiles table.

# COMMAND ----------

def psub3_rebaseline(state, sbu, project_id, total_months, project_duration_days, rng):
    """PSUB 3: Rebaseline decision with profiled delays and SOW date shifts.
    
    Hard preconditions (from plan + your schedule generator):
    - days_behind > 14 (P25 of genuine rebaseline triggers)
    - months_since_rebaseline >= 3 (minimum inter-rebaseline gap)
    - pct_complete < 0.92 (<2% of real rebaselines occur above this)
    - n_rebaselines < max
    
    Probability ramp:
    P(rb) = min(0.85, p0(phi) + α·(days_behind/30) + β·1[open_rfis excess] + γ·1[BPI<0.90])
    
    When triggered:
    - Delay category sampled from your DELAY_CATEGORY_PMF
    - Delay magnitude from your schedule constraints with slack-conditional capping
    - SOW date shifts from your profiled date_shift_profiles
    - Task count perturbation from profiled ratio (always from V0 base, never compounding)
    """
    state.is_rebaseline_month = False
    state.rebaseline_delay_category = None

    # ── Hard preconditions ──
    if state.days_behind <= 14: return state
    if state.months_since_rebaseline is not None and state.months_since_rebaseline < 3: return state
    if state.pct_complete >= 0.92: return state
    if state.n_rebaselines >= state.n_rebaselines_max: return state

    # ── Probability ramp (SPI + BPI driven) ──
    p0 = 0.30 * (1.0 - state.phi)
    exp_rfis = max(1.0, state.rfi_rate_3m * 3)
    p = min(0.85,
        p0 +
        0.018 * (state.days_behind / 30.0) +
        0.08 * (1 if state.open_rfis > exp_rfis else 0) +
        0.10 * (1 if state.bpi < 0.90 else 0)
    )
    if rng.random() >= p: return state

    # ═══════ REBASELINE TRIGGERED ═══════
    state.is_rebaseline_month = True
    state.n_rebaselines += 1
    state.months_since_rebaseline = 0
    state.rebaseline_months.append(state.sim_month)

    # ── Sample delay category from your PMF ──
    categories = list(DELAY_CATEGORY_PMF.keys())
    probs = np.array(list(DELAY_CATEGORY_PMF.values())); probs /= probs.sum()
    delay_category = rng.choice(categories, p=probs)
    state.rebaseline_delay_category = delay_category

    # ── Sample delay magnitude (from your schedule constraints) ──
    # With slack-conditional capping (from your schedule generator)
    slack_remaining = max(30, days_between(state.dt_snapshot, state.current_tco))
    bounds = hier_lookup_bounds(sched_constr_df, f'delay_{delay_category}', sbu=sbu)
    if bounds:
        delay_raw = sample_from_bounds(bounds)
        p95 = bounds.get('p95', bounds.get('p75', delay_raw))
        p99 = bounds.get('p99', bounds.get('max', delay_raw))
        # Slack-based cap (from your generator: SLACK_MULTIPLIER = 0.9)
        slack_cap = slack_remaining * 0.9
        if delay_raw > slack_cap:
            delay_raw = slack_cap * (1 + rng.uniform(0, 0.20))  # SLACK_MARGIN
        # Soft p95 cap, hard p99 cap
        if delay_raw > p95:
            delay_raw = p95 * (1 + rng.uniform(0, 0.20))
        delay_days = max(7, min(int(delay_raw), int(p99) if pd.notna(p99) else 999))
    else:
        # Fallback defaults by category (from your generator)
        defaults = {'minimal': 4, 'small': 17, 'moderate': 41, 'major': 81, 'large': 149, 'severe': 275}
        delay_days = defaults.get(delay_category, 30)

    # ── Apply delay to TCO (monotonically increasing) ──
    state.current_tco = state.current_tco + timedelta(days=delay_days)
    state.cat_schedule_code = generate_schedule_code(project_id, state.n_rebaselines)

    # ── SOW date shifts (from your profiled date_shift_profiles) ──
    # Uses same logic as your task summary generator: completion-aware shifts
    # with drift damping to prevent compounding across rebaselines
    n_sows = len(state.sow_states)
    project_timing = state.sim_month / max(1, total_months)
    delay_scale = max(0.2, min(3.0, delay_days / MEDIAN_DELAY_DAYS))

    for sow in state.sow_states:
        pct = sow.pct_complete * 100  # convert to 0-100 scale
        normalized_pos = sow.sow_sequence_order / max(n_sows, 1)

        if pct < 95:
            comp_bucket = get_completion_bucket(pct)
            date_pos_bucket = get_date_shift_position_bucket(normalized_pos)
            shifts = lookup_date_shift(comp_bucket, date_pos_bucket)

            # Finish shift — consistent regardless of completion, apply directly
            actual_finish_shift = int(shifts['finish_shift'] * min(delay_scale, 1.5))

            # Start shift — scale by remaining work (from your generator)
            remaining_work = max(0.05, 1.0 - pct / 100.0)
            capped_delay_scale = min(delay_scale, 1.5)
            raw_start_shift = int(shifts['start_shift'] * capped_delay_scale * remaining_work)

            # Drift damping (from your generator: max 180-day ceiling)
            v0_start = sow.v0_sow_start or sow.dt_sow_start
            max_drift = 180
            current_drift = max(0, (sow.dt_sow_start - v0_start).days) if sow.dt_sow_start and v0_start else 0
            remaining_budget = max(0, max_drift - current_drift)
            damping = remaining_budget / max_drift if max_drift > 0 else 0
            dampened_start = int(raw_start_shift * damping)

            # Apply shifts
            new_finish = sow.dt_sow_finish + timedelta(days=actual_finish_shift)
            new_start = sow.dt_sow_start + timedelta(days=dampened_start)
            # Hard floor: never before construction start
            if new_start < state.dt_snapshot - timedelta(days=state.sim_month * 30):
                new_start = state.dt_snapshot - timedelta(days=state.sim_month * 30)

            sow.dt_sow_finish = new_finish
            sow.dt_sow_start = new_start

            # Enforce min duration
            min_dur = SOW_MIN_DURATIONS.get(sow.cat_sow, SOW_MIN_DURATION_DEFAULT)
            raw_dur = (sow.dt_sow_finish - sow.dt_sow_start).days
            if raw_dur < min_dur:
                sow.dt_sow_finish = sow.dt_sow_start + timedelta(days=min_dur)
            sow.sow_duration_days = max(min_dur, (sow.dt_sow_finish - sow.dt_sow_start).days)

        # Task count perturbation (from V0 base, never compounding — from your generator)
        task_ratio_bounds = hier_lookup_bounds(task_constr_df, 'task_count_change_ratio', sbu=sbu)
        if task_ratio_bounds:
            ratio = np.clip(sample_from_bounds(task_ratio_bounds), 0.5, 2.0)
        else:
            ratio = rng.uniform(0.85, 1.15)
        sow.n_tasks_total = max(5, int(sow.n_tasks_base * ratio))

    # ── Partial SPI recovery after rebaseline ──
    state.spi = min(1.05, state.spi + min(0.05, 0.03 * (1 + state.phi)))
    state.days_behind = max(0, state.days_behind - int(delay_days * 0.6))

    return state

# COMMAND ----------

print("✓ Cell 3 loaded: PSUBs 1-3 (Schedule, Shocks, Rebaseline)")

✓ Loaded SOW profiles from analysts.self_managed.synthetic_task_summary_by_sow_v1 (58 SOWs)
    (15) Tower Crane 1 - Usage Duration: avg_tasks=9, std=1
    (09) Buys: avg_tasks=61, std=20
    (9BX) Set Generator: avg_tasks=98, std=4
Running 1 simulation (seed=42)...

═══════════════════════════════════════════════════════
RUN COMPLETE — SYN-COM-TEST-001-RUN-00-S42
═══════════════════════════════════════════════════════
  Phi:                0.425
  Duration:           45 months
  Final SPI:          0.995
  Final BPI:          0.862
  EAC/Contract:       1.0800
  Delay (days):       6
  Rebaselines:        0
  Total RFIs:         933
  Total COs:          84
  Shocks:             0
  Runtime:            1457ms
═══════════════════════════════════════════════════════
Generating LLM text for 5 sample RFIs...

  SYN-COM-TEST-001-RFI-0001 | A SUBSTRUCTURE | closed
    Subject: Clarification on Existing Utility Location Marks Prior to Excavation
    Description: Field crew encountered unmark

sim_month,pct_complete,spi,bpi,days_behind,open_rfis,n_rebaselines,is_rebaseline_month,cat_schedule_code,composite_risk
0,0.009642113974162491,1.5,1.5,0,0,0,false,SYN-COM-TEST-001-000.01-BL,0.0
1,0.016661830556542787,1.5,1.5,0,0,0,false,SYN-COM-TEST-001-000.01-BL,0.0
2,0.02343186129164861,1.5,1.5,0,0,0,false,SYN-COM-TEST-001-000.01-BL,0.0
3,0.030416470303859855,1.3950085750298815,1.5,0,0,0,false,SYN-COM-TEST-001-000.01-BL,0.0
4,0.05220327415383079,1.4111099959363653,1.5,0,0,0,false,SYN-COM-TEST-001-000.01-BL,0.0
5,0.060819706580164515,1.1523908930065312,1.5,0,0,0,false,SYN-COM-TEST-001-000.01-BL,0.0
6,0.07362850896044397,1.0286449755802447,1.3009569052811931,0,0,0,false,SYN-COM-TEST-001-000.01-BL,0.0
7,0.07362850896044397,0.8365326887789406,1.0893367164504626,41,0,0,false,SYN-COM-TEST-001-000.01-BL,0.06538692448842376
8,0.0848090579512773,0.730479016390799,0.8880584347039784,88,0,0,false,SYN-COM-TEST-001-000.01-BL,0.15258501956208903
9,0.11308562882700825,0.7597570014812912,0.8848065736596751,85,0,0,false,SYN-COM-TEST-001-000.01-BL,0.1421745699436135


sim_month,bpi,spi,eac_contract_ratio,cumulative_spend,monthly_burn,is_at_risk,is_distressed
0,1.5,1.5,0.4827336536985326,4499.46374722821,4499.46374722821,false,false
1,1.5,1.5,0.4827336536985326,109636.14267298585,105136.67892575765,false,false
2,1.5,1.5,0.4827336536985326,441483.54083709436,331847.3981641085,false,false
3,1.5,1.3950085750298815,0.4827336536985326,1793094.243959008,1351610.7031219136,false,false
4,1.5,1.4111099959363653,0.4827336536985326,3144889.541577696,1351795.2976186879,false,false
5,1.5,1.1523908930065312,0.5945070893301936,5423662.009933504,2278772.4683558084,false,false
6,1.3009569052811931,1.0286449755802447,0.7421208984538853,8196188.283231694,2772526.2732981904,false,false
7,1.0893367164504626,0.8365326887789406,0.8862891453277932,9788422.241746748,1592233.9585150536,false,false
8,0.8880584347039784,0.730479016390799,1.08,1.3830233961528953E7,4041811.719782206,true,true
9,0.8848065736596751,0.7597570014812912,1.08,1.8509211078303408E7,4678977.116774456,true,true


id_rfi_synth,sim_month,cat_trade_l1,cat_sow,cat_status,is_near_rebaseline,cat_schedule_code,str_rfi_subject
SYN-COM-TEST-001-RFI-0001,0,A SUBSTRUCTURE,(02) Sitework,closed,false,SYN-COM-TEST-001-000.01-BL,Clarification on Existing Utility Location Marks Prior to Excavation
SYN-COM-TEST-001-RFI-0002,0,A SUBSTRUCTURE,(03) Concrete / Structure,closed,false,SYN-COM-TEST-001-000.01-BL,Concrete Pour Schedule Conflict with Weather Forecast Next Week
SYN-COM-TEST-001-RFI-0003,0,A SUBSTRUCTURE,(03) Concrete / Structure,closed,false,SYN-COM-TEST-001-000.01-BL,Concrete Compressive Strength Testing Schedule and Cylinder Location Clarification
SYN-COM-TEST-001-RFI-0004,0,A SUBSTRUCTURE,(03) Concrete / Structure,closed,false,SYN-COM-TEST-001-000.01-BL,Clarification Required: Foundation Concrete Strength Testing Schedule and Procedures
SYN-COM-TEST-001-RFI-0005,0,A SUBSTRUCTURE,(03) Concrete / Structure,closed,false,SYN-COM-TEST-001-000.01-BL,Clarification Required: Concrete Strength Testing Frequency for Foundation Pours
SYN-COM-TEST-001-RFI-0006,0,G SITEWORK,(02) Sitework,closed,false,SYN-COM-TEST-001-000.01-BL,null
SYN-COM-TEST-001-RFI-0007,0,A SUBSTRUCTURE,(03) Concrete / Structure,closed,false,SYN-COM-TEST-001-000.01-BL,null
SYN-COM-TEST-001-RFI-0008,0,A SUBSTRUCTURE,(02) Sitework,closed,false,SYN-COM-TEST-001-000.01-BL,null
SYN-COM-TEST-001-RFI-0009,0,G SITEWORK,(02) Sitework,closed,false,SYN-COM-TEST-001-000.01-BL,null
SYN-COM-TEST-001-RFI-0010,0,A SUBSTRUCTURE,(02) Sitework,closed,false,SYN-COM-TEST-001-000.01-BL,null


In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC ## Cell 4: PSUBs 4-7 (RFI, Change Orders, Budget, Guardrails)
# MAGIC Division-level budget with S-curves, CO rates, cost decomposition from your budget generator.

# COMMAND ----------

# MAGIC %md
# MAGIC ### PSUB 4 — RFI Generation
# MAGIC Profiled trade weights, lifecycle timing, resolution days, status distribution.
# MAGIC Proper open RFI backlog tracking with probabilistic resolution.

# COMMAND ----------

def psub4_rfis(state, sbu, size_cat, project_id, total_months, total_project_rfis, rng):
    """PSUB 4: RFI generation with profiled distributions and backlog tracking.
    
    λ_t = λ_base(phi) × m_phase × m_pressure × m_rb × m_close × m_shock
    
    Resolution: existing open RFIs resolve probabilistically each month
    based on profiled median resolution time, so backlog builds up
    and feeds back into SPI dampening (PSUB 1).
    """
    # ── Monthly arrival rate ──
    base_monthly = total_project_rfis / max(1, total_months)
    lam = phi_modulate(base_monthly, state.phi, 0.75, 1.25)

    lp = state.pct_complete
    # Phase multiplier (lifecycle-dependent)
    if lp < 0.10: phase_m = 0.4
    elif lp < 0.30: phase_m = 0.8
    elif lp < 0.75: phase_m = 1.2
    elif lp < 0.90: phase_m = 1.0
    else: phase_m = 0.5

    # Pressure multiplier — ramps with composite risk (pre-rebaseline buildup)
    m_pressure = 1.0 + 0.5 * state.composite_risk

    # Post-rebaseline spike (from your profiled rebaseline_spike_ratio)
    m_rb = 1.0
    if state.months_since_rebaseline is not None and state.months_since_rebaseline <= 1:
        m_rb = float(rb_spike_bounds.get('median', 1.08))

    # BPI-driven
    if state.bpi > 1.05: m_bpi = 0.85
    elif state.bpi < 0.90: m_bpi = 1.15
    else: m_bpi = 1.0

    # Closeout dampener
    m_close = 0.70 if lp > 0.85 else 1.0

    # Shock multiplier
    m_shock = state.get_rfi_shock_multiplier()

    n_new = rng.poisson(max(0.1, lam * phase_m * m_pressure * m_rb * m_bpi * m_close * m_shock))

    # ── Generate individual RFI records ──
    project_sows = {s.cat_sow for s in state.sow_states}
    rfis = []

    for _ in range(n_new):
        state.total_rfis += 1
        trade = sample_trade(sbu, rng)
        lc_pct = sample_rfi_lifecycle_pct(trade, rng)

        # Map trade → SOW (from your TRADE_TO_SOW)
        candidates = TRADE_TO_SOW.get(trade, ['(33) NO CODE'])
        available = [s for s in candidates if any(s in ps for ps in project_sows)]
        cat_sow = rng.choice(available) if available else (
            rng.choice(list(project_sows)) if project_sows else rng.choice(candidates))

        dt_c = state.dt_snapshot + timedelta(days=int(rng.integers(0, 28)))
        due_days = random.choice(DUE_DATE_OPTIONS)
        dt_due = dt_c + timedelta(days=due_days)
        status = sample_rfi_status(sbu, rng)

        # Resolution timing (from your profiled resolution_days distribution)
        dt_resolved = None
        if status == 'closed':
            res_days = sample_resolution_days(rng)
            dt_resolved = dt_c + timedelta(days=res_days)
        elif status == 'open' and lp >= 0.70 and rng.random() < 0.15:
            status = 'overdue'

        design_issue = rng.random() < rfi_design_issue_pct.get(trade, 0.35)
        is_near_rb = (state.months_since_rebaseline is not None and state.months_since_rebaseline <= 1)

        # SOW timing zone (from your RFI generator's profiled sow_timing_zone_pct)
        sow_timing = "During"
        for s in state.sow_states:
            if s.cat_sow == cat_sow and state.dt_snapshot and s.dt_sow_start and s.dt_sow_finish:
                if state.dt_snapshot < s.dt_sow_start: sow_timing = "Pre-SOW"
                elif state.dt_snapshot > s.dt_sow_finish: sow_timing = "Post-SOW"
                break

        rfis.append({
            'sim_run_id': None, 'ids_project_number_synth': project_id,
            'id_rfi_synth': generate_rfi_id(project_id, state.total_rfis),
            'n_rfi_sequence': state.total_rfis, 'parent_rfi_id': None,
            'dt_rfi_created': dt_c, 'dt_rfi_due': dt_due, 'dt_rfi_resolved': dt_resolved,
            'cat_status': status, 'cat_trade_l1': trade, 'cat_sow': cat_sow,
            'sow_sequence_timing': sow_timing, 'cat_design_issue': design_issue,
            'sim_month': state.sim_month, 'cat_schedule_code': state.cat_schedule_code,
            'is_near_rebaseline': is_near_rb, 'pct_lifecycle_at_creation': lp,
            'str_rfi_subject': None, 'str_rfi_description': None, 'str_rfi_response': None,
            'is_shock_related': len(state.active_shocks) > 0,
            'cat_shock_type': state.active_shocks[0].shock_type if state.active_shocks else None,
        })

    # ── Chain follow-ups (~10% of closed, from your RFI generator) ──
    for r in [r for r in rfis if r['cat_status'] == 'closed']:
        if rng.random() < 0.10:
            state.total_rfis += 1
            fu = r.copy()
            fu['id_rfi_synth'] = generate_rfi_id(project_id, state.total_rfis)
            fu['n_rfi_sequence'] = state.total_rfis
            fu['parent_rfi_id'] = r['id_rfi_synth']
            fu['cat_status'] = 'open'; fu['dt_rfi_resolved'] = None
            fu['dt_rfi_created'] = (r['dt_rfi_resolved'] + timedelta(days=int(rng.integers(1, 7)))) if r['dt_rfi_resolved'] else r['dt_rfi_created'] + timedelta(days=7)
            rfis.append(fu)

    # ── Update open RFI backlog (KEY FIX: proper resolution tracking) ──
    new_open = sum(1 for r in rfis if r['cat_status'] in ('open', 'overdue', 'draft'))
    # Existing open RFIs resolve probabilistically based on profiled resolution time
    # median ~10 days = ~3 resolutions per month per open RFI
    med_res = float(rfi_resolution_bounds.get('median', 10))
    resolve_prob_per_rfi = min(0.95, 30.0 / max(1, med_res))  # monthly resolution probability
    resolved_this_month = sum(1 for _ in range(state.open_rfis) if rng.random() < resolve_prob_per_rfi)
    state.open_rfis = max(0, state.open_rfis + new_open - resolved_this_month)

    # Rolling 3-month rate
    state.rfi_history.append(n_new)
    if len(state.rfi_history) > 3: state.rfi_history = state.rfi_history[-3:]
    state.rfi_rate_3m = sum(state.rfi_history) / len(state.rfi_history) if state.rfi_history else 0

    return state, rfis

# COMMAND ----------

# MAGIC %md
# MAGIC ### PSUB 5 — Change Orders
# MAGIC Division-level CO rates from your budget generator.
# MAGIC Rebaseline-driven COs use your DELAY_CO_MULTIPLIER by severity.

# COMMAND ----------

def psub5_change_orders(state, sbu, project_id, rng):
    """PSUB 5: Change orders using your profiled division-level CO rates.
    
    From your budget generator:
    - Rebaseline months: CO_PROB_REBASE_MONTH = 0.8227, with delay multiplier by severity
    - Normal months: CO_PROB_NORMAL_MONTH = 0.6261, with RFI-density multiplier for MEP divs
    - CO values from profiled p50_co_in_ratio and p50_co_out_ratio
    """
    cos = []
    co_counter = state.n_approved_cos + state.n_pending_cos

    is_rb_month = state.is_rebaseline_month
    delay_cat = state.rebaseline_delay_category
    delay_mult = DELAY_CO_MULTIPLIER.get(delay_cat, 1.0) if delay_cat else 1.0

    # Process each budget line (division-level)
    for bl in state.budget_lines:
        div = bl.division
        co_r = get_co_rate(div, sbu)

        if is_rb_month:
            # Rebaseline-driven CO (from your budget generator)
            if rng.random() < CO_PROB_REBASE_MONTH:
                base_ratio = float(co_r.get('p50_co_in_ratio', 0.001))
                co_in = bl.amt_original * base_ratio * delay_mult * rng.uniform(0.8, 1.2)
                bl.cum_approved_in += co_in

                co_counter += 1
                cos.append({
                    'sim_run_id': None, 'ids_project_number_synth': project_id,
                    'id_co_synth': generate_co_id(project_id, co_counter),
                    'sim_month': state.sim_month,
                    'dt_co_created': state.dt_snapshot + timedelta(days=int(rng.integers(0, 28))),
                    'cat_csi_division': div, 'amt_co_value': co_in,
                    'cat_approval_status': 'approved',
                    'cat_budget_category': bl.category,
                    'is_rebaseline_driven': True,
                    'cat_schedule_code': state.cat_schedule_code,
                })

            # Contingency out (from your budget generator)
            if bl.category == 'contingency' and div not in MANDATORY_DIVISIONS:
                base_out = float(co_r.get('p50_co_out_ratio', 0.001))
                co_out = bl.amt_original * base_out * delay_mult * rng.uniform(0.8, 1.2)
                bl.cum_approved_out -= co_out
        else:
            # Normal month CO (from your budget generator)
            rfi_mult = 1.15 if div in {"03", "05", "21", "22", "23", "26"} else 1.0
            if rng.random() < CO_PROB_NORMAL_MONTH * rfi_mult:
                base_ratio = float(co_r.get('p50_co_in_ratio', 0.001))
                co_in = base_ratio * bl.amt_original * rng.uniform(0.5, 1.5)
                bl.cum_approved_in += co_in

                co_counter += 1
                st = 'approved' if rng.random() < 0.75 else ('pending' if rng.random() < 0.5 else 'rejected')
                cos.append({
                    'sim_run_id': None, 'ids_project_number_synth': project_id,
                    'id_co_synth': generate_co_id(project_id, co_counter),
                    'sim_month': state.sim_month,
                    'dt_co_created': state.dt_snapshot + timedelta(days=int(rng.integers(0, 28))),
                    'cat_csi_division': div, 'amt_co_value': co_in,
                    'cat_approval_status': st,
                    'cat_budget_category': bl.category,
                    'is_rebaseline_driven': False,
                    'cat_schedule_code': state.cat_schedule_code,
                })

            # CO out
            if rng.random() < float(co_r.get('prob_co_out', 0.02)):
                base_out = float(co_r.get('p50_co_out_ratio', 0.001))
                bl.cum_approved_out -= base_out * bl.amt_original * rng.uniform(0.5, 1.5)

    # Update state totals
    for co in cos:
        if co['cat_approval_status'] == 'approved':
            state.n_approved_cos += 1; state.amt_approved_co_value += co['amt_co_value']
        elif co['cat_approval_status'] == 'pending':
            state.n_pending_cos += 1; state.amt_pending_co_value += co['amt_co_value']

    return state, cos

# COMMAND ----------

# MAGIC %md
# MAGIC ### PSUB 6 — Budget Burn
# MAGIC Per-division S-curve burn with cost decomposition (direct/committed/projected)
# MAGIC from your budget generator. JTD monotonically non-decreasing per line.

# COMMAND ----------

def psub6_budget(state, sbu, total_months, project_duration_days, rng):
    """PSUB 6: Division-level budget burn using your profiled S-curves.
    
    From your budget generator:
    - Each division has its own S-curve shape (interpolate_budget_scurve)
    - Division lifecycle windows gate when spend occurs
    - Category lifecycle offsets shift timing (materials lead, allowance lags)
    - Cost decomposition: direct_ratio, committed_ratio from profiled ratios
    - JTD is monotonically non-decreasing per budget line
    - ECAC = max(projected, jtd + remaining * efficiency)
    """
    lifecycle_pct = min(1.0, max(0.0, state.sim_month / max(1, total_months)))
    total_burn = 0.0

    for bl in state.budget_lines:
        div = bl.division
        # Division lifecycle window gating (from your budget generator)
        win_start, win_end = DIV_LIFECYCLE_WINDOW.get(div, (0.0, 1.0))
        is_in_window = (win_start <= lifecycle_pct <= win_end) or div in MANDATORY_DIVISIONS

        if not is_in_window:
            continue  # No spend outside lifecycle window

        # Revised budget after COs
        amt_revised = bl.amt_original + bl.cum_approved_in + bl.cum_approved_out

        # S-curve blend: 70% SOW-grounded + 30% category-shifted (from your generator)
        base_share = interpolate_budget_scurve(div, lifecycle_pct)
        cat_offset = CATEGORY_LIFECYCLE_OFFSET.get(bl.category, 0.0)
        adj_lp = min(1.0, max(0.0, lifecycle_pct + cat_offset))
        cat_share = interpolate_budget_scurve(div, adj_lp)
        cume_share = min(1.0, max(0.0, 0.7 * base_share + 0.3 * cat_share))

        # JTD burns against revised scope, monotonically non-decreasing
        amt_jtd = cume_share * amt_revised
        amt_jtd = max(amt_jtd, bl.jtd_max)
        bl.jtd_max = amt_jtd

        # Monthly burn = delta from last month's JTD
        # (positive only — can't un-spend)
        # We approximate by: this month's incremental = jtd - previous jtd
        # Since jtd_max is cumulative, the monthly is the delta
        monthly_this_line = max(0, amt_jtd - (bl.jtd_max - (amt_jtd - bl.jtd_max) if bl.jtd_max > amt_jtd else 0))
        # Simpler: just accumulate jtd across all lines
        total_burn += amt_jtd

    # Total cumulative spend = sum of all JTDs across divisions
    prev_spend = state.cumulative_spend
    state.cumulative_spend = total_burn
    state.monthly_burn = max(0, state.cumulative_spend - prev_spend)
    state.actual_cost = state.cumulative_spend

    # Earned value
    state.earned_value = state.pct_complete * state.amt_budget_total

    # BPI (= CPI = EV / AC)
    if state.actual_cost > 0:
        state.bpi = min(2.0, state.earned_value / state.actual_cost)
    else:
        state.bpi = 1.0

    # EAC with efficiency (from your budget generator)
    # efficiency = max(0.70, 0.95 - 0.25 * lifecycle_pct)
    eff = max(0.70, 0.95 - 0.25 * lifecycle_pct)
    if state.bpi > 0:
        state.eac = max(state.amt_budget_total,
                        state.cumulative_spend + max(0, state.amt_budget_total - state.cumulative_spend) * eff / state.bpi)
    else:
        state.eac = state.amt_budget_total * 1.5

    return state

# COMMAND ----------

# MAGIC %md
# MAGIC ### PSUB 7 — Guardrails & State Commit
# MAGIC Hard bounds from real Platinum data. Retrospective rebaseline audit.

# COMMAND ----------

def psub7_guardrails(state, prev_pct):
    """PSUB 7: Enforce realism guardrails and recompute composite risk."""
    # pct_complete: [0, 1], monotonically non-decreasing
    state.pct_complete = max(prev_pct, min(1.0, state.pct_complete))
    # TCO monotonically increasing
    if state.current_tco and state.original_tco and state.current_tco < state.original_tco:
        state.current_tco = state.original_tco
    # EAC ceiling (P99 from real Platinum)
    state.eac = min(state.eac, 1.08 * state.amt_contract)
    # Monthly burn ceiling (P99 from real Platinum)
    state.monthly_burn = min(state.monthly_burn, 0.08 * state.amt_contract)
    # SPI / BPI bounds
    state.spi = max(0.50, min(1.50, state.spi))
    state.bpi = max(0.50, min(1.50, state.bpi))
    # Rebaseline count ceiling
    state.n_rebaselines = min(state.n_rebaselines, state.n_rebaselines_max)
    # Composite risk
    state.compute_composite_risk()
    # Increment months since rebaseline
    if state.months_since_rebaseline is not None and not state.is_rebaseline_month:
        state.months_since_rebaseline += 1
    return state


def audit_rebaselines(sched_snapshots, final_state):
    """Retrospective rebaseline audit — 6 months post-rebaseline.
    
    Three indicators (from plan Section 7):
    1. SPI recovery above 0.92 within 3 months → maybe premature
    2. TCO stability for 4+ months → likely justified
    3. Open RFI reduction ≥ 20% within 2 months → resolved pressure
    """
    flags = []
    for rb_month in final_state.rebaseline_months:
        post = [s for s in sched_snapshots if rb_month < s['sim_month'] <= rb_month + 6]
        if len(post) < 3:
            flags.append('inconclusive'); continue
        spi_recovered = any(s['spi'] > 0.92 for s in post[:3])
        tcos = [str(s.get('current_tco')) for s in post[:4] if s.get('current_tco')]
        tco_stable = len(set(tcos)) <= 1
        rb_snap = next((s for s in sched_snapshots if s['sim_month'] == rb_month), None)
        rfi_at_rb = rb_snap['open_rfis'] if rb_snap else 0
        rfi_resolved = False
        if rfi_at_rb > 0 and len(post) >= 2:
            rfi_2m = min(s['open_rfis'] for s in post[:2])
            rfi_resolved = (rfi_at_rb - rfi_2m) / rfi_at_rb >= 0.20
        if spi_recovered and not tco_stable and not rfi_resolved: flags.append('premature')
        elif tco_stable and (rfi_resolved or not spi_recovered): flags.append('justified')
        elif rfi_resolved: flags.append('justified')
        else: flags.append('inconclusive')
    return flags

# COMMAND ----------

print("✓ Cell 4 loaded: PSUBs 4-7 (RFI, CO, Budget, Guardrails)")

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC ## Cell 5: Simulator, LLM, Runner
# MAGIC Ties all PSUBs together. Single run for testing, then scale to K=10.

# COMMAND ----------

# MAGIC %md
# MAGIC ### 5.1 ProjectSimulator

# COMMAND ----------

class ProjectSimulator:
    def __init__(self, project_id, cat_sbu, cat_region, cat_main_asset_class,
                 amt_contract, gross_area, dt_construction_start, planned_duration_months):
        self.project_id = project_id; self.sbu = cat_sbu; self.region = cat_region
        self.asset_class = cat_main_asset_class; self.amt_contract = amt_contract
        self.gross_area = gross_area; self.dt_start = dt_construction_start
        self.planned_months = planned_duration_months
        self.size_cat = get_size_category(gross_area)
        self.project_duration_days = planned_duration_months * 30

    def run(self, n_runs=1, base_seed=None):
        if base_seed is None: base_seed = np.random.default_rng().integers(0, 2**31)
        all_s, all_b, all_r, all_c, all_reg = [], [], [], [], []
        for ri in range(n_runs):
            seed = base_seed + ri; t0 = time.time()
            res = self._single_run(ri, seed)
            ms = int((time.time() - t0) * 1000)
            run_id = generate_run_id(self.project_id, ri, seed)
            for r in res['sched']: r['sim_run_id'] = run_id; r['random_seed'] = seed
            for r in res['budg']: r['sim_run_id'] = run_id; r['random_seed'] = seed
            for r in res['rfis']: r['sim_run_id'] = run_id
            for r in res['cos']: r['sim_run_id'] = run_id
            all_s.extend(res['sched']); all_b.extend(res['budg'])
            all_r.extend(res['rfis']); all_c.extend(res['cos'])
            fs = res['final']
            all_reg.append({
                'sim_run_id': run_id, 'ids_project_number_synth': self.project_id,
                'random_seed': seed, 'phi': res['phi'],
                'total_duration_months': fs.sim_month, 'total_delay_days': fs.days_behind,
                'n_rebaselines': fs.n_rebaselines, 'final_eac': fs.eac,
                'final_eac_contract_ratio': fs.eac / max(1, fs.amt_contract),
                'total_rfis': fs.total_rfis,
                'total_change_orders': fs.n_approved_cos + fs.n_pending_cos,
                'final_spi': fs.spi, 'final_bpi': fs.bpi,
                'rebaseline_months': fs.rebaseline_months,
                'sim_audit_justified_flags': res.get('audit_flags', []),
                'shock_events_fired': res.get('shocks', []),
                'total_runtime_ms': ms,
            })
        return {
            'schedule_snapshots': all_s, 'budget_snapshots': all_b,
            'rfi_events': all_r, 'change_orders': all_c, 'run_registry': all_reg,
        }

    def _single_run(self, ri, seed):
        rng = np.random.default_rng(seed)
        init = ProjectInitializer(
            self.project_id, self.sbu, self.region, self.asset_class,
            self.amt_contract, self.gross_area, self.dt_start, self.planned_months, rng
        )
        state = init.initialize()
        # Store sbu on state for PSUB 1 completion lookup
        state._sbu = self.sbu
        tm = self.planned_months
        pdur = self.project_duration_days
        sc = init.generate_shock_calendar(tm + 12)

        # Pre-sample total RFI count for this project run (from your RFI generator)
        total_project_rfis = sample_n_rfis_for_project(self.sbu, self.size_cat, rng)

        sched, budg, rfis, cos, shocks = [], [], [], [], []
        for m in range(tm + 24):
            state.sim_month = m
            state.dt_snapshot = add_months(self.dt_start, m)
            pp = state.pct_complete

            # ── 7 PSUBs in sequence ──
            state = psub1_schedule(state, tm, pdur, rng)
            state = psub2_shocks(state, sc, rng)
            state = psub3_rebaseline(state, self.sbu, self.project_id, tm, pdur, rng)
            state, nr = psub4_rfis(state, self.sbu, self.size_cat, self.project_id, tm, total_project_rfis, rng)
            rfis.extend(nr)
            state, nc = psub5_change_orders(state, self.sbu, self.project_id, rng)
            cos.extend(nc)
            state = psub6_budget(state, self.sbu, tm, pdur, rng)
            state = psub7_guardrails(state, pp)

            er = state.eac / max(1, state.amt_contract)

            # ── Collect schedule snapshot ──
            sched.append({
                'ids_project_number_synth': self.project_id, 'sim_month': m,
                'dt_snapshot': state.dt_snapshot,
                'pct_complete': state.pct_complete, 'spi': state.spi, 'bpi': state.bpi,
                'days_behind': state.days_behind,
                'current_tco': state.current_tco, 'original_tco': state.original_tco,
                'pct_float_consumed': state.pct_float_consumed,
                'n_rebaselines': state.n_rebaselines,
                'months_since_rebaseline': state.months_since_rebaseline,
                'is_rebaseline_month': state.is_rebaseline_month,
                'cat_schedule_code': state.cat_schedule_code,
                'composite_risk': state.composite_risk,
                'open_rfis': state.open_rfis, 'total_rfis': state.total_rfis,
            })

            # ── Collect budget snapshot ──
            budg.append({
                'ids_project_number_synth': self.project_id, 'sim_month': m,
                'dt_snapshot': state.dt_snapshot,
                'planned_value': state.planned_value,
                'earned_value': state.earned_value, 'actual_cost': state.actual_cost,
                'bpi': state.bpi, 'spi': state.spi,
                'amt_budget_total': state.amt_budget_total, 'amt_contract': state.amt_contract,
                'eac': state.eac, 'eac_contract_ratio': er,
                'cumulative_spend': state.cumulative_spend,
                'monthly_burn': state.monthly_burn,
                'n_approved_cos': state.n_approved_cos,
                'amt_approved_co_value': state.amt_approved_co_value,
                'n_pending_cos': state.n_pending_cos,
                'amt_pending_co_value': state.amt_pending_co_value,
                'is_at_risk': er > 1.00, 'is_distressed': er > 1.02,
                'n_budget_divisions': len(set(bl.division for bl in state.budget_lines)),
            })

            # ── Track shocks ──
            for sh in [s for s in sc if s['month'] == m]:
                shocks.append({
                    'sim_month': m, 'cat_shock_type': sh['type'],
                    'magnitude': sh.get('spi_penalty', 0),
                })

            # ── Termination check ──
            if state.pct_complete >= 0.995:
                break

        # Retrospective rebaseline audit
        audit_flags = audit_rebaselines(sched, state)

        return {
            'sched': sched, 'budg': budg, 'rfis': rfis, 'cos': cos,
            'final': state, 'phi': state.phi, 'shocks': shocks,
            'audit_flags': audit_flags,
        }

# COMMAND ----------

# MAGIC %md
# MAGIC ### 5.2 LLM Client (simple version for testing)

# COMMAND ----------

_api_key = dbutils.secrets.get(scope="llm-access-tokens", key="anthropic-api-key")

def generate_rfi_text(trade, sow, lifecycle_pct, is_rebaseline, shock_type=None):
    """Simple LLM text generation for testing. Will swap in full diversity engine later."""
    phase = "early" if lifecycle_pct < 0.30 else ("mid" if lifecycle_pct < 0.70 else "late")
    ctx = f"Trade: {trade}, SOW: {sow}, Phase: {phase} ({lifecycle_pct:.0%} complete)"
    if is_rebaseline: ctx += ", recently rebaselined schedule"
    if shock_type: ctx += f", active {shock_type} event"
    prompt = f"""You are writing a realistic construction RFI for a Suffolk Construction project.
Context: {ctx}
Return ONLY a JSON object (no markdown, no backticks):
{{"subject": "10-15 word RFI subject line", "description": "50-80 word field description of the issue", "response": "30-50 word resolution or null if unresolved"}}"""
    try:
        resp = requests.post("https://api.anthropic.com/v1/messages",
            headers={"x-api-key": _api_key, "anthropic-version": "2023-06-01", "content-type": "application/json"},
            json={"model": "claude-haiku-4-5-20251001", "max_tokens": 300,
                  "messages": [{"role": "user", "content": prompt}]}, timeout=10)
        if resp.status_code == 200:
            text = resp.json()["content"][0]["text"].strip().replace("```json","").replace("```","").strip()
            d = json.loads(text)
            return {"str_rfi_subject": d.get("subject"), "str_rfi_description": d.get("description"),
                    "str_rfi_response": d.get("response")}
    except Exception as e:
        print(f"  LLM failed: {e}")
    return {"str_rfi_subject": None, "str_rfi_description": None, "str_rfi_response": None}

# COMMAND ----------

# MAGIC %md
# MAGIC ### 5.3 Run Single Simulation

# COMMAND ----------

project_id = "SYN-COM-TEST-001"
sim = ProjectSimulator(
    project_id=project_id, cat_sbu="commercial", cat_region="northeast",
    cat_main_asset_class="Office", amt_contract=150_000_000, gross_area=250_000,
    dt_construction_start=date(2024, 3, 1), planned_duration_months=36,
)

print("Running 1 simulation (seed=42)...")
results = sim.run(n_runs=1, base_seed=42)
reg = results['run_registry'][0]
print(f"\n{'═'*60}")
print(f"RUN COMPLETE — {reg['sim_run_id']}")
print(f"{'═'*60}")
for k in ['phi','total_duration_months','final_spi','final_bpi','final_eac_contract_ratio',
          'total_delay_days','n_rebaselines','total_rfis','total_change_orders','total_runtime_ms']:
    v = reg[k]
    print(f"  {k:.<40} {v:.3f}" if isinstance(v, float) else f"  {k:.<40} {v}")
print(f"  {'shocks':.<40} {len(reg['shock_events_fired'])}")
if reg['sim_audit_justified_flags']:
    print(f"  {'rebaseline_audit':.<40} {reg['sim_audit_justified_flags']}")
print(f"{'═'*60}")

# COMMAND ----------

# MAGIC %md
# MAGIC ### 5.4 LLM Text for Sample RFIs

# COMMAND ----------

sample = results['rfi_events'][:5]
print(f"Generating LLM text for {len(sample)} sample RFIs...\n")
for rfi in sample:
    out = generate_rfi_text(rfi['cat_trade_l1'], rfi['cat_sow'],
                            rfi['pct_lifecycle_at_creation'],
                            rfi['is_near_rebaseline'], rfi['cat_shock_type'])
    rfi.update(out)
    print(f"  {rfi['id_rfi_synth']} | {rfi['cat_trade_l1']} → {rfi['cat_sow']} | {rfi['cat_status']}")
    print(f"    Subject: {rfi['str_rfi_subject']}")
    desc = rfi['str_rfi_description']
    print(f"    Desc: {desc[:80]}..." if desc and len(desc) > 80 else f"    Desc: {desc}")
    print()

# COMMAND ----------

# MAGIC %md
# MAGIC ### 5.5 Preview as Spark DataFrames

# COMMAND ----------

sched_df = spark.createDataFrame(pd.DataFrame(results['schedule_snapshots']))
budg_df  = spark.createDataFrame(pd.DataFrame(results['budget_snapshots']))
rfi_df   = spark.createDataFrame(pd.DataFrame(results['rfi_events']))
co_df    = spark.createDataFrame(pd.DataFrame(results['change_orders']))

print(f"Schedule: {sched_df.count()} | Budget: {budg_df.count()} | "
      f"RFIs: {rfi_df.count()} | COs: {co_df.count()}")

# COMMAND ----------

display(sched_df.select("sim_month","pct_complete","spi","bpi","days_behind",
                         "open_rfis","n_rebaselines","is_rebaseline_month",
                         "composite_risk").orderBy("sim_month"))

# COMMAND ----------

display(budg_df.select("sim_month","bpi","spi","eac_contract_ratio","cumulative_spend",
                        "monthly_burn","n_budget_divisions","is_at_risk","is_distressed"
                        ).orderBy("sim_month"))

# COMMAND ----------

display(rfi_df.select("id_rfi_synth","sim_month","cat_trade_l1","cat_sow","cat_status",
                       "sow_sequence_timing","is_near_rebaseline","parent_rfi_id",
                       "str_rfi_subject").limit(25))

# COMMAND ----------

display(co_df.select("id_co_synth","sim_month","cat_csi_division","cat_budget_category",
                      "amt_co_value","cat_approval_status","is_rebaseline_driven").limit(25))

# COMMAND ----------

# MAGIC %md
# MAGIC ### 5.6 Write to Delta (uncomment when ready)

# COMMAND ----------

# sched_df.write.format("delta").mode("overwrite").option("mergeSchema","true").saveAsTable(OUT_SCHEDULE_SNAPSHOTS)
# budg_df.write.format("delta").mode("overwrite").option("mergeSchema","true").saveAsTable(OUT_BUDGET_SNAPSHOTS)
# rfi_df.write.format("delta").mode("overwrite").option("mergeSchema","true").saveAsTable(OUT_RFI_EVENTS)
# co_df.write.format("delta").mode("overwrite").option("mergeSchema","true").saveAsTable(OUT_CHANGE_ORDERS)
# spark.createDataFrame(pd.DataFrame(results['run_registry'])).write.format("delta").mode("overwrite").option("mergeSchema","true").saveAsTable(OUT_RUN_REGISTRY)
# print(f"✓ 5 tables written to {OUTPUT_CATALOG}")